# **MC2融合算子开发流程讲解**

本章将以 Matmul+ReduceScatter 融合算子为例，详细讲解 MC2（Matrix Computation & Communication）融合算子的完整开发流程，涵盖以下内容：
- 环境准备
- 算子分析（业务背景、融合方案、Tiling策略）
- 工程结构与Tiling数据设计
- Host侧代码实现详解
- Kernel侧代码实现详解
- 编译与运行
- 测试与验证
- 课后实践
---

## **1. 环境准备**

> **运行环境说明**：本章节内容基于 **Ascend 950** 平台开发与验证，采用 **`<<<>>>` Kernel 直调（Direct Invoke）** 方式启动算子，不使用 msopgen 工具或 aclnn 接口。请确保运行环境为 950 系列硬件。

本文所有内容均存放于Sources文件夹。在开始之前，先对jupyter环境进行初始化，导入CANN环境变量并创建代码目录。

In [ ]:
!mkdir -p Sources/01.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\nEnvironment initialization completed successfully!")

由于本项目是 Kernel 直调工程（不使用 msopgen 生成），需要手动创建目录结构：

In [ ]:
# 创建工程目录结构
!mkdir -p Sources/01.03/mte_matmul_reduce_scatter/op_host/scripts
!mkdir -p Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/kernel
!mkdir -p Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/block
!mkdir -p Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/tile
!mkdir -p Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/tiling
!mkdir -p Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/utils

print("Directory structure created successfully!")

---
## **2. 算子分析**

### **2.1 算子背景**

Matmul+ReduceScatter 是分布式大模型训练中的高频算子组合，广泛用于 Megatron-LM、DeepSpeed 等框架的张量并行（Tensor Parallelism）场景：

- **Matmul（矩阵乘法）**：每个设备对自身的输入数据 X 和共享权重 W 执行矩阵乘法
- **ReduceScatter（规约分散）**：将所有设备的 Matmul 输出按行切分，各设备只保留自己负责的那份聚合结果

| 步骤 | 操作 | 执行核 |
|------|------|--------|
| 1 | Matmul：Y_i = X_i[M,K] × W^T[K,N]，各 Rank 独立计算完整 Matmul 输出[M,N] | AIC |
| 2 | Matmul 结果直接写入各 Rank 的 Win Area（不写 GM） | AIC |
| 3 | 状态同步：所有 Rank 完成 Matmul 后通知对方 | AIV |
| 4 | ReduceScatter：从各 Rank 的 Win Area 读取对应 chunk，求和后输出 | AIV |

最终每个 Rank i 得到 ReduceScatter 输出的第 i 个 chunk：**Z_i = Σ_j(Y_j 的第 i 个 chunk)**。

### **2.2 关键技术要点**

| 技术点 | 说明 |
|--------|------|
| **Split-M Tiling** | 将 M 维度按 rankSize 和 splitMCnt 分块，平衡各核计算负载 |
| **GET 模式通信** | Matmul 先完成计算写入 Win Area，ReduceScatter 再从各 Rank 读取 |
| **Win Area 机制** | 利用 MTE 引擎直接访问其他 Rank 的 Win Area 内存，无需 Host 端中转 |
| **AIC/AIV 混合调度** | AIC 核执行 Matmul 计算，AIV 核执行 ReduceScatter 通信 |
| **状态同步** | 通过 Win Area 的 Status 区域做 Flag 同步 |

### **2.3 Split-M Tiling 策略**

| 参数 | 含义 | 计算方式（M=1024, rankSize=4, minSplitM=128） |
|------|------|------|
| `chunkM` | 每个Rank负责的M行数 | `1024 / 4 = 256` |
| `splitMCnt` | M维度细粒度切分因子 | `256 / 128 = 2` |
| `splitM` | 每轮Matmul计算的M大小 | `256 / 2 = 128` |
| `baseM/N/K` | 单次MMAD指令的基础块大小 | `min(splitM, 128) = 128` |

---
## **3. 工程结构与Tiling数据设计**

### **3.1 工程目录结构**

本项目采用 Kernel 直调（Direct Invoke）方式，工程目录结构如下：

In [ ]:
!find Sources/01.03/mte_matmul_reduce_scatter -type d | sort | sed 's;[^/]*/;|____;g;s;____|;    |;g'

核心文件职责：

```
mte_matmul_reduce_scatter/
├── CMakeLists.txt                          # CMake构建配置
├── build.sh                                # 一键编译脚本
├── run_multi_data.sh                       # 多尺寸自动化测试脚本
├── op_host/
│   ├── mte_matmul_reduce_scatter_host.cpp  # Host侧主程序
│   ├── host_utils.h                        # Host侧工具函数
│   └── scripts/ (gen_data.py, verify_result.py)
└── op_kernel/
    ├── mte_matmul_reduce_scatter_kernel.cpp # Kernel入口
    ├── matmul_kernel_utils.h                # Kernel工具函数
    └── include/ (kernel/, block/, tiling/, tile/, utils/)
```

### **3.2 Tiling数据结构定义**

首先定义 Tiling 结构体，它们负责在 Host 和 Device 之间传递算子的维度信息和切分策略。

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/tiling/grouped_matmul_fp16_tiling_data.h
#ifndef GROUPED_MATMUL_FP16_TILING_DATA_H
#define GROUPED_MATMUL_FP16_TILING_DATA_H

#include <cstdint>

struct GroupedMatmulFp16TilingData {
    uint32_t groupNum = 0;
    uint32_t maxM = 0;
    uint32_t n = 0;
    uint32_t k = 0;
    uint32_t baseM = 0;
    uint32_t baseN = 0;
    uint32_t baseK = 0;
    uint32_t kAL1 = 0;
    uint32_t kBL1 = 0;
    uint32_t usedCoreNum = 0;
    uint8_t dbL0C = 1;
};

#endif


In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/tiling/matmul_reduce_scatter_tiling.h
#ifndef MATMUL_REDUCE_SCATTER_TILING_H
#define MATMUL_REDUCE_SCATTER_TILING_H

#include <stdint.h>

// 对于AscendC编译（Device），使用kernel_tiling.h中的类型定义
// 对于Host编译（g++），需要从pkg_inc/hccl路径获取兼容定义
#if defined(__ASCENDC__) || defined(__CCE__)
#include <kernel_tiling/kernel_tiling.h>
using Mc2InitTilingType = ::Mc2InitTiling;
using Mc2CcTilingType = ::Mc2CcTiling;
#else
// Host编译时的定义（与hccl_tilingdata.h中一致）
#pragma pack(push, 8)
typedef struct {
    uint8_t reserved[64];
} Mc2InitTilingType;
typedef struct {
    uint8_t reserved[280];
} Mc2CcTilingType;
#pragma pack(pop)
#endif

namespace MatmulReduceScatterImpl {

struct MatmulReduceScatterTilingInfo {
    uint64_t perRankLen;       // Matmul输出大小 (M * N)
    uint64_t chunkSize;        // ReduceScatter输出大小 = perRankLen / rankDim
    uint32_t m;                // M维度
    uint32_t n;                // N维度
    uint32_t k;                // K维度
    uint32_t aicNum;           // AIC核数量
    uint32_t baseM;            // Matmul基础M切分
    uint32_t baseN;            // Matmul基础N切分
    uint32_t baseK;            // Matmul基础K切分
    uint32_t splitMCnt;        // M切分因子（如2）
    uint32_t splitM;           // 每个split块的M = M/rankSize/splitMCnt
    uint32_t chunkM;           // 每个chunk的M = M/rankSize
};

struct MatmulReduceScatterTilingData {
    Mc2InitTilingType mc2InitTiling;
    Mc2CcTilingType mc2CcTiling;
    MatmulReduceScatterTilingInfo tilingInfo;
};

}

#endif

**关键 Tiling 参数说明：**

- `MatmulReduceScatterTilingInfo`：包含 `perRankLen`（完整Matmul输出大小）、`chunkSize`（ReduceScatter输出大小）、`m/n/k`、`aicNum`、`baseM/N/K`、`splitMCnt/splitM/chunkM`
- `MatmulReduceScatterTilingData`：包含 MC2 通信 Tiling（`Mc2InitTiling` + `Mc2CcTiling`）和算子 TilingInfo
- `GroupedMatmulFp16TilingData`：Matmul 模块专用参数（`groupNum`、`maxM`、`n/k`、`baseM/N/K`、`usedCoreNum`、`dbL0C`）

---
## **4. 工具函数与常量定义**

按照从底层到顶层的顺序，先完成基础工具函数的编写。

### **4.1 Kernel 工具函数**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/matmul_kernel_utils.h
#ifndef MTE_MATMUL_REDUCE_SCATTER_KERNEL_UTILS_H
#define MTE_MATMUL_REDUCE_SCATTER_KERNEL_UTILS_H

#if ASC_DEVKIT_MAJOR >= 9
#include "kernel_basic_intf.h"
#include "std/algorithm.h"
#else
#include "kernel_operator.h"
#endif
#include "lib/matmul_intf.h"
#include "lib/std/tuple.h"

namespace AscendC {
namespace Std {
template <typename...>
struct always_false : public false_type {};

template <typename... Tp>
constexpr bool always_false_v = always_false<Tp...>::value;
} // namespace Std
} // namespace AscendC

template <int32_t t>
using Int = AscendC::Std::integral_constant<int32_t, t>;

using _0 = Int<0>;

constexpr static int64_t L0A_SIZE = 64 * 1024;
constexpr static int64_t L0B_SIZE = 64 * 1024;
constexpr static int64_t L0C_SIZE = 256 * 1024;
constexpr static int64_t L1_SIZE = 512 * 1024;
constexpr static int32_t BT_SIZE = 4096;

constexpr int MNK_M = 0;
constexpr int MNK_N = 1;
constexpr int MNK_K = 2;
constexpr int MNK_B = 3;
constexpr int MNK_M0 = 4;
constexpr int MNK_N0 = 5;

struct MatmulShape {
    int64_t m;
    int64_t n;
    int64_t k;
    int64_t b;
};

template <typename T>
__aicore__ inline T Max(T a, T b)
{
    return a > b ? a : b;
}

template <typename T>
__aicore__ inline T Min(T a, T b)
{
    return a > b ? b : a;
}

__aicore__ inline uint64_t CeilDiv(uint64_t a, uint64_t b)
{
    if (b == 0) {
        return a;
    }
    return (a + b - 1) / b;
}

__host_aicore__ inline int64_t CeilDiv(int64_t a, int64_t b)
{
    if (b == 0) {
        return a;
    }
    return (a + b - 1) / b;
}

__host_aicore__ inline int64_t CeilAlign(int64_t a, int64_t b)
{
    if (b == 0) {
        return a;
    }
    return (a + b - 1) / b * b;
}

__aicore__ inline uint64_t Align(uint64_t a, uint64_t b)
{
    if (b == 0) {
        return a;
    }
    return (a + b - 1) / b * b;
}

#include "matmul/matmul_config.h"
#include "matmul/tiling.h"
#include "../../impl/adv_api/detail/matmul/utils/matmul_utils.h"

namespace layout {
struct RowMajor {};
struct ColumnMajor {};
} // namespace layout

template <typename T>
struct TagToFormat {
    static_assert(AscendC::Std::always_false_v<T>, "TagToFormat is not implemented for this layout");
};

template <>
struct TagToFormat<layout::RowMajor> {
    using tag = layout::RowMajor;
    static constexpr CubeFormat format = CubeFormat::ND;
};

template <>
struct TagToFormat<layout::ColumnMajor> {
    using tag = layout::ColumnMajor;
    static constexpr CubeFormat format = CubeFormat::ND;
};

template <typename T>
struct TagToTrans {
    static_assert(AscendC::Std::always_false_v<T>, "TagToTrans is not implemented for this layout");
};

template <>
struct TagToTrans<layout::RowMajor> {
    static constexpr bool value = false;
};

template <>
struct TagToTrans<layout::ColumnMajor> {
    static constexpr bool value = true;
};

template <size_t I, typename T>
__aicore__ constexpr inline decltype(auto) Get(T&& t)
{
    return AscendC::Std::get<I>(AscendC::Std::forward<T>(t));
}

template <size_t First, size_t Second, size_t... Rest, typename T>
__aicore__ constexpr inline decltype(auto) Get(T&& t)
{
    return Get<Second, Rest...>(AscendC::Std::get<First>(AscendC::Std::forward<T>(t)));
}

struct KernelMultiBlockOnKAxisWithScale {};

struct QuantMatmulMxMultiBlockMmad {
    using ScheduleType = KernelMultiBlockOnKAxisWithScale;
};

#endif // MTE_MATMUL_REDUCE_SCATTER_KERNEL_UTILS_H

该文件定义了：
- **内存常量**：`L0A_SIZE`（64KB）、`L0B_SIZE`（64KB）、`L0C_SIZE`（256KB）、`L1_SIZE`（512KB）
- **布局标签**：`layout::RowMajor` / `layout::ColumnMajor`，用于描述矩阵的内存排列方式
- **工具函数**：`CeilDiv`、`CeilAlign`、`Align`、`Max`、`Min`
- **转置标识**：`TagToTrans` 将布局映射为是否需要转置
- **调度策略**：`QuantMatmulMxMultiBlockMmad` 作为 Split-M 多 Block 的调度标识

### **4.2 常量定义**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/utils/grouped_matmul_fp16_constant.h
#ifndef GROUPED_MATMUL_FP16_CONSTANT_H
#define GROUPED_MATMUL_FP16_CONSTANT_H

#include <cstdint>

namespace GroupedMatmulFp16Recipe {
constexpr int32_t MNK_M = 0;
constexpr int32_t MNK_N = 1;
constexpr int32_t MNK_K = 2;

constexpr int8_t GROUP_TYPE_SPLIT_M = 0;
constexpr int8_t GROUP_TYPE_NO_SPLIT = -1;

constexpr uint64_t CUBE_BLOCK = 16UL;
constexpr uint64_t DOUBLE_BUFFER = 2UL;
constexpr uint8_t MTE1_MTE2_EVENT_ID_NUM = 4;
constexpr static uint32_t FINAL_ACCUMULATION = 3U;
constexpr static uint32_t NON_FINAL_ACCUMULATION = 2U;
} // namespace GroupedMatmulFp16Recipe

#endif


### **4.3 ReduceScatter 工具函数**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/utils/reduce_scatter_utils.h
#ifndef UTILS_REDUCE_SCATTER_H
#define UTILS_REDUCE_SCATTER_H

#include "kernel_operator.h"

namespace MatmulReduceScatterImpl {

using namespace AscendC;

__aicore__ inline uint64_t FloorAlign(uint64_t a, uint32_t b)
{
    uint64_t bTemp = static_cast<uint64_t>(b);
    return (bTemp == 0) ? a : (a / bTemp) * bTemp;
}

__aicore__ inline uint64_t MIN(uint64_t a, uint64_t b)
{
    return (a < b) ? a : b;
}

__aicore__ inline uint64_t BlockAlignMod(uint64_t a, uint32_t b)
{
    if (b == 0) {
        return 0;
    }
    uint64_t c = a % b;
    return c ? c : b;
}

template<AscendC::HardEvent event>
__aicore__ inline void SyncFunc()
{
    AscendC::TEventID eventID = GetTPipePtr()->FetchEventID<event>();
    AscendC::SetFlag<event>(eventID);
    AscendC::WaitFlag<event>(eventID);
}

}

#endif

### **4.4 MMAD Tile 封装**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/tile/tile_mmad_std.h
#ifndef GROUPED_MATMUL_FP16_TILE_MMAD_H
#define GROUPED_MATMUL_FP16_TILE_MMAD_H

#include "impl/atom/cube_compute/mmad.h"

namespace Tile {

struct MmadStd {
    template <typename Tp, const Tp& traits, typename T, typename U, typename S>
    __aicore__ inline static void Mad(
        const T& dst, const U& fm, const S& filter, uint16_t m, uint16_t k, uint16_t n, uint8_t unitFlagCtrl,
        bool disableGemv, bool cmatrixSource, bool cmatrixInitVal)
    {
        mad(dst.Data().Get(), fm.Data().Get(), filter.Data().Get(), m, k, n, unitFlagCtrl, disableGemv,
            cmatrixSource, cmatrixInitVal);
    }
};

} // namespace Tile

namespace AscendC {
namespace Te {

template <typename Opration, typename TraitStruct>
struct MmadTraits<Opration, TraitStruct> {
    using TraitType = typename TraitStruct::TraitType;
    static constexpr const TraitType defaultTrait = TraitStruct::value;

    template <const TraitType& trait = defaultTrait, typename... Args>
    __aicore__ inline void MmadUnpack(const Args&... args) const
    {
        Opration::template Mad<TraitType, trait, Args...>(args..., m, k, n, unitFlagCtrl, disableGemv, cmatrixSource, cmatrixInitVal);
    }

    uint16_t m = 0;
    uint16_t k = 0;
    uint16_t n = 0;
    uint8_t unitFlagCtrl = 0;
    bool disableGemv = false;
    bool cmatrixSource = false;
    bool cmatrixInitVal = false;
};

template <>
struct MmadTraits<::Tile::MmadStd>
    : public MmadTraits<::Tile::MmadStd, MmadTraitDefault, ::Tile::MmadStd, MmadTraitDefault> {};

} // namespace Te
} // namespace AscendC

#endif


`Tile::MmadStd` 封装了硬件 MMAD 指令的调用接口，通过 `MmadTraits` trait 类实现默认参数（m/k/n维度、unitFlagCtrl等）的自动填充。

---
## **5. Matmul 计算模块**

Matmul 模块负责在 AIC 核上执行 Split-M 矩阵乘法。代码分为 Block 调度器和 MMAD 计算块两层。

### **5.1 Block 调度器工具**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/block/block_scheduler_utils.h
/*!
 * \file block_scheduler_utils.h
 * \brief Scheduler traits shared by the grouped MXFP4 recipe wrappers.
 */
#ifndef GROUPED_MATMUL_BLOCK_SCHEDULER_UTILS_H
#define GROUPED_MATMUL_BLOCK_SCHEDULER_UTILS_H

#include "matmul_kernel_utils.h"

namespace Block {

template <size_t N, typename Tp>
__aicore__ constexpr inline decltype(auto) GetIntegralConstant()
{
    static_assert(AscendC::Std::is_tuple_v<Tp>, "Input must be a tuple type");
    return AscendC::Std::tuple_element<N, Tp>::type::value;
}

__aicore__ inline bool IsMTail(int64_t mTileIdx, int64_t mTileNum)
{
    return (mTileIdx - (mTileNum - 1)) % mTileNum == 0;
}

__aicore__ inline bool IsNTail(int64_t nTileIdx, int64_t nTileNum)
{
    return nTileIdx == (nTileNum - 1);
}


} // namespace Block

#endif


### **5.2 Block 调度器（Split-M）**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/block/block_scheduler_fp16_split_m.h
#ifndef GROUPED_MATMUL_FP16_BLOCK_SCHEDULER_SPLIT_M_H
#define GROUPED_MATMUL_FP16_BLOCK_SCHEDULER_SPLIT_M_H
#include "./block_scheduler_utils.h"

namespace Block {
constexpr int64_t INNER_AXIS_MIN_SPLIT_VAL = 128LL;
constexpr int64_t WINDOW_LEN = 4LL;

// ============================================================
// BlockSchedulerFp16SplitM: M维度Split Block调度器
// 将 [M, N] 问题空间以 baseM x baseN 为Tile单位分配
// GetTileIdx: 返回当前Block的Tile坐标 [mTileIdx, nTileIdx]
// GetBlockShape: 返回Tile的实际尺寸(含尾块截断和细粒度Split)
// ============================================================
template <class ProblemShape_, bool TransA_, bool TransB_>
class BlockSchedulerFp16SplitM {
public:
    using TupleShape = AscendC::Shape<int64_t, int64_t, int64_t, int64_t>;
    using BlockCoord = AscendC::Coord<int64_t, int64_t, int64_t, int64_t>;
    using ProblemShape = ProblemShape_;

private:
    int64_t mCnt_;
    int64_t nCnt_;
    int64_t totalCnt_;
    int64_t m_{0};
    int64_t n_{0};
    int64_t k_;
    int64_t mTailCnt_{1};
    int64_t nTailCnt_{1};
    int64_t mTailAlign_{1};
    int64_t nTailAlign_{1};
    int64_t tailCnt_{1};
    int64_t mainMWindow_;
    int64_t tailWindow_;
    int64_t mainRow_;
    int64_t round_;
    int64_t roundIdx_;
    int32_t baseM_;
    int32_t baseN_;
    int32_t baseK_;
    int32_t mBaseTail_;
    int32_t nBaseTail_;
    uint32_t blockNum_;
    uint32_t blockIdx_ = AscendC::GetBlockIdx() / AscendC::GetTaskRation();
    uint32_t startBlockIdx_;
    uint32_t endBlockIdx_;

public:
    __aicore__ inline BlockSchedulerFp16SplitM(int32_t baseM, int32_t baseN, int32_t baseK, uint32_t usedCoreNum)
        : baseM_(baseM), baseN_(baseN), baseK_(baseK), blockNum_(usedCoreNum), endBlockIdx_(usedCoreNum - 1)
    {
    }

    __aicore__ inline BlockSchedulerFp16SplitM(int32_t baseM, int32_t baseN, int32_t baseK)
        : baseM_(baseM), baseN_(baseN), baseK_(baseK), blockNum_(AscendC::GetBlockNum()), endBlockIdx_(AscendC::GetBlockNum() - 1)
    {
    }

    __aicore__ inline void UpdateNextProblem(const TupleShape &problemShape)
    {
        k_ = Get<MNK_K>(problemShape);
        if (m_ != Get<MNK_M>(problemShape) || n_ != Get<MNK_N>(problemShape)) {
            m_ = Get<MNK_M>(problemShape);
            n_ = Get<MNK_N>(problemShape);
            mCnt_ = CeilDiv(m_, baseM_);
            nCnt_ = CeilDiv(n_, baseN_);
            mBaseTail_ = m_ - (mCnt_ - 1) * baseM_;
            nBaseTail_ = n_ - (nCnt_ - 1) * baseN_;
            totalCnt_ = mCnt_ * nCnt_;
            mainMWindow_ = WINDOW_LEN < mCnt_ ? WINDOW_LEN : mCnt_;
            mainRow_ = mCnt_ / mainMWindow_ - 1;
            tailWindow_ = mCnt_ - mainMWindow_ * mainRow_;
        }
        roundIdx_ = 0;
        round_ = CeilDiv(totalCnt_, blockNum_);
        startBlockIdx_ = endBlockIdx_ == blockNum_ - 1 ? 0 : (endBlockIdx_ + 1);
        endBlockIdx_ = (totalCnt_ + startBlockIdx_ - 1) % blockNum_;
        if (startBlockIdx_ > endBlockIdx_ && (blockIdx_ > endBlockIdx_ && blockIdx_ < startBlockIdx_)) {
            round_ -= 1;
        } else if (startBlockIdx_ <= endBlockIdx_ && (blockIdx_ > endBlockIdx_ || blockIdx_ < startBlockIdx_)) {
            round_ -= 1;
        }
    }

    __aicore__ inline void UpdateBaseM(uint32_t baseM)
    {
        baseM_ = baseM;
    }

    __aicore__ inline void SetTailAlign(uint32_t mTailAlign, uint32_t nTailAlign)
    {
        mTailAlign_ = mTailAlign;
        nTailAlign_ = nTailAlign;
    }

    __aicore__ inline int64_t GetTailTileCnt()
    {
        return Min(static_cast<int64_t>(endBlockIdx_ + 1), totalCnt_);
    }

    __aicore__ inline void UpdateTailTile(uint32_t mTailCnt, uint32_t nTailCnt)
    {
        mTailCnt_ = mTailCnt;
        nTailCnt_ = nTailCnt;
        tailCnt_ = mTailCnt * nTailCnt;
        int64_t tailOriCnt = GetTailTileCnt();
        int64_t newEndBlockIdx = endBlockIdx_ + tailOriCnt * (tailCnt_ - 1);
        if (blockIdx_ > endBlockIdx_ && blockIdx_ <= newEndBlockIdx) {
            round_ += 1;
        }
        if (blockIdx_ > newEndBlockIdx) {
            mTailCnt_ = 1;
            nTailCnt_ = 1;
            tailCnt_ = 1;
        }
        endBlockIdx_ = newEndBlockIdx;
    }

    __aicore__ inline void UpdateTailTile()
    {
        int64_t remainTile = (AscendC::GetBlockNum() - endBlockIdx_ - 1) / GetTailTileCnt() + 1;
        if (remainTile <= 1) {
            return;
        }
        int64_t mMin = AscendC::BLOCK_CUBE;
        int64_t nMin = AscendC::BLOCK_CUBE;
        if constexpr (TransA_) {
            mMin = INNER_AXIS_MIN_SPLIT_VAL;
        }
        if constexpr (!TransB_) {
            nMin = INNER_AXIS_MIN_SPLIT_VAL;
        }
        int64_t mTile = Min(CeilDiv(mBaseTail_, mMin), remainTile);
        int64_t nTile = Min(CeilDiv(nBaseTail_, nMin), remainTile);
        while (mTile * nTile > remainTile) {
            if (mTile >= nTile) {
                mTile -= 1;
            } else {
                nTile -= 1;
            }
        }
        UpdateTailTile(mTile, nTile);
    }

    // GetTileIdx: 多Block Tile坐标分配
    // 采用行优先(RowMajor) + 奇行反序(zigzag)策略均衡负载:
    //   行0: [0,1,2,...,nCnt_-1], 行1: [nCnt_-1,...,1,0], ...
    // 返回false表示当前Block已无更多任务
    __aicore__ inline bool GetTileIdx(BlockCoord &blockCoord)
    {
        if (round_ == 0 || roundIdx_ > round_ - 1) {
            return false;
        }
        int64_t newBlockIdx = (roundIdx_ == round_ - 1) ? blockIdx_ / tailCnt_ : blockIdx_;
        int64_t index = newBlockIdx + roundIdx_ * blockNum_;
        if (blockIdx_ < startBlockIdx_) {
            index += blockNum_ - startBlockIdx_;
        } else if (endBlockIdx_ + 1 >= tailCnt_ * totalCnt_) {
            index -= startBlockIdx_ / tailCnt_;
        } else {
            index -= startBlockIdx_;
        }
        int64_t rowIdx = index / nCnt_ / mainMWindow_;
        if (rowIdx < mainRow_) {
            Get<MNK_M>(blockCoord) = rowIdx * mainMWindow_ + index % mainMWindow_;
            Get<MNK_N>(blockCoord) = (index / mainMWindow_) % nCnt_;
        } else {
            rowIdx = mainRow_;
            int64_t tailIndex = index - mainRow_ * mainMWindow_ * nCnt_;
            Get<MNK_M>(blockCoord) = mainRow_ * mainMWindow_ + tailIndex % tailWindow_;
            Get<MNK_N>(blockCoord) = (tailIndex / tailWindow_) % nCnt_;
        }
        if (rowIdx & 1) {
            Get<MNK_N>(blockCoord) = nCnt_ - 1 - Get<MNK_N>(blockCoord);
        }
        roundIdx_++;
        return true;
    }

    // GetBlockShape: 根据Tile坐标返回实际M/N尺寸
    // 尾块处理: 最后一行/列可能需要截断为 baseM % m 或 baseN % n
    // 细粒度Split: 非尾轮时可按 mTailCnt_ × nTailCnt_ 进一步细分Tile
    __aicore__ inline TupleShape GetBlockShape(const BlockCoord &blockCoord)
    {
        int64_t singleCoreM = Get<MNK_M>(blockCoord) != (mCnt_ - 1) ? baseM_ : mBaseTail_;
        int64_t singleCoreN = Get<MNK_N>(blockCoord) != (nCnt_ - 1) ? baseN_ : nBaseTail_;
        if (tailCnt_ == 1 || roundIdx_ < round_) {
            return {singleCoreM, singleCoreN, 0, 0};
        }
        int64_t singleCoreMSplit = (singleCoreM + mTailCnt_ - 1) / mTailCnt_;
        int64_t singleCoreNSplit = (singleCoreN + nTailCnt_ - 1) / nTailCnt_;
        if constexpr (TransA_) {
            singleCoreMSplit = Align(singleCoreMSplit, INNER_AXIS_MIN_SPLIT_VAL);
        }
        singleCoreNSplit = Align(singleCoreNSplit, nTailAlign_);
        if constexpr (!TransB_) {
            singleCoreNSplit = Align(singleCoreNSplit, INNER_AXIS_MIN_SPLIT_VAL);
        }
        int64_t mSplitIdx = (blockIdx_ % tailCnt_) % mTailCnt_;
        int64_t nSplitIdx = (blockIdx_ % tailCnt_) / mTailCnt_;
        int64_t mSplitAddrOffset = mSplitIdx * singleCoreMSplit;
        int64_t nSplitAddrOffset = nSplitIdx * singleCoreNSplit;
        if (mSplitAddrOffset >= singleCoreM || nSplitAddrOffset >= singleCoreN) {
            return {0, 0, 0, 0};
        }
        singleCoreM = Min(singleCoreM - mSplitAddrOffset, singleCoreMSplit);
        singleCoreN = Min(singleCoreN - nSplitAddrOffset, singleCoreNSplit);
        return {singleCoreM, singleCoreN, mSplitAddrOffset, nSplitAddrOffset};
    }

    __aicore__ inline int64_t GetEndBlockIdx() const
    {
        return endBlockIdx_;
    }
};

} // namespace Block
#endif


`BlockSchedulerFp16SplitM` 负责将 M 维度的工作分配到各 AIC 核：

- 以 `baseM`（如 128）为最小分块单位
- 通过 `GetTileIdx()` 为每个核分配独立的 Tile 坐标
- `GetBlockShape()` 返回当前核需要处理的 [M, N, K] 大小
- 处理 M 和 N 维度的尾块（tail）对齐

### **5.3 MMAD 计算块（核心）**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/block/block_mmad_fp16_split_m.h
#ifndef GROUPED_MATMUL_FP16_BLOCK_MMAD_SPLIT_M_H
#define GROUPED_MATMUL_FP16_BLOCK_MMAD_SPLIT_M_H

#include "matmul_kernel_utils.h"
#include "experimental/tensor_api/tensor.h"
#include "../utils/grouped_matmul_fp16_constant.h"

namespace Block {

namespace {
constexpr static uint64_t HALF_L0C_SIZE = L0C_SIZE / GroupedMatmulFp16Recipe::DOUBLE_BUFFER / sizeof(float);
constexpr static uint16_t INPUT_BUFFER_MTE1_MTE2_BASE = 2;
} // namespace

template <
    class DispatchPolicy_,
    class AType_, class LayoutA_,
    class BType_, class LayoutB_,
    class CType_, class LayoutC_,
    class Enable = void>
class BlockMmadFp16 {
    static_assert(AscendC::Std::always_false_v<DispatchPolicy_>, "Should not be here!");
};

template <
    class DispatchPolicy_,
    class AType_, class LayoutA_,
    class BType_, class LayoutB_,
    class CType_, class LayoutC_>
class BlockMmadFp16<
    DispatchPolicy_, AType_, LayoutA_, BType_, LayoutB_, CType_, LayoutC_,
    AscendC::Std::enable_if_t<
        AscendC::Std::is_base_of_v<QuantMatmulMxMultiBlockMmad, DispatchPolicy_>>> {
// ============================================================
// BlockMmadFp16: 单Block Matmul计算单元
// 数据流: GM -> L1(双缓冲) -> L0A/L0B -> MMAD -> L0C(双缓冲) -> FixPipe -> GM
// 核心特性: Ping-Pong双缓冲实现搬运与计算Overlap
// ============================================================
public:
    using AType = AType_;
    using BType = BType_;
    using CType = CType_;
    using LayoutA = LayoutA_;
    using LayoutB = LayoutB_;
    using DispatchPolicy = DispatchPolicy_;
    using TupleShape = AscendC::Shape<int64_t, int64_t, int64_t>;
    using BlockShape = AscendC::Shape<int64_t, int64_t, int64_t>;
    static constexpr bool transA = ::TagToTrans<LayoutA>::value;
    static constexpr bool transB = ::TagToTrans<LayoutB>::value;
    static constexpr CubeFormat formatB = ::TagToFormat<LayoutB>::format;
    static_assert(!transA, "BlockMmadFp16 only supports non-transposed A.");
    static_assert(!transB, "BlockMmadFp16 only supports non-transposed B (ND).");
    constexpr static uint64_t HALF_L0_SIZE = L0A_SIZE / GroupedMatmulFp16Recipe::DOUBLE_BUFFER / sizeof(AType);
    using MakeLayoutAL1 = AscendC::Te::NzLayoutFormat<AType>;
    using MakeLayoutBL1 = AscendC::Te::NzLayoutFormat<BType>;

    struct Params {
        GM_ADDR aGmAddr{nullptr};
        GM_ADDR bGmAddr{nullptr};
        GM_ADDR groupListGmAddr{nullptr};
        GM_ADDR cGmAddr{nullptr};
    };

    struct L1Params {
        uint64_t kAL1;
        uint64_t kBL1;
    };

    __aicore__ inline BlockMmadFp16()
    {
        #pragma unroll
        for (uint8_t i = 0; i < GroupedMatmulFp16Recipe::MTE1_MTE2_EVENT_ID_NUM; ++i) {
            AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(i);
        }
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(0);
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(1);
    }

    __aicore__ inline ~BlockMmadFp16()
    {
        #pragma unroll
        for (uint8_t i = 0; i < GroupedMatmulFp16Recipe::MTE1_MTE2_EVENT_ID_NUM; ++i) {
            AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(i);
        }
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(0);
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(1);
    }

    // 初始化：从Tiling参数中提取M/N/K维度，在L1内存中分配A/B双缓冲区池
    __aicore__ inline void Init(
        const TupleShape& problemShape, const BlockShape& l0TileShape, const L1Params& l1Params, bool enableL0cPingPong)
    {
        constexpr uint64_t l1BufNum = GroupedMatmulFp16Recipe::DOUBLE_BUFFER;

        m_ = Get<GroupedMatmulFp16Recipe::MNK_M>(problemShape);
        n_ = Get<GroupedMatmulFp16Recipe::MNK_N>(problemShape);
        k_ = Get<GroupedMatmulFp16Recipe::MNK_K>(problemShape);
        kAL1_ = l1Params.kAL1;
        kBL1_ = l1Params.kBL1;
        baseM_ = Get<GroupedMatmulFp16Recipe::MNK_M>(l0TileShape);
        baseN_ = Get<GroupedMatmulFp16Recipe::MNK_N>(l0TileShape);
        baseK_ = Get<GroupedMatmulFp16Recipe::MNK_K>(l0TileShape);
        orderAL1BL1_ = l1Params.kAL1 >= l1Params.kBL1;
        enableL0cPingPong_ = enableL0cPingPong;

        bL1OneBuffer_ = baseN_ * Align(kBL1_, GroupedMatmulFp16Recipe::CUBE_BLOCK) * sizeof(bfloat16_t);
        aL1OneBuffer_ = baseM_ * Align(kAL1_, GroupedMatmulFp16Recipe::CUBE_BLOCK) * sizeof(bfloat16_t);

        for (int32_t bufferId = 0; bufferId < static_cast<int32_t>(l1BufNum); bufferId++) {
            uint64_t l1Offset = (L1_SIZE >> 1) * (bufferId & 1);
            l1BufferAOffset_[bufferId] = l1Offset + aL1OneBuffer_ * (bufferId >> 1);
            l1BufferBOffset_[bufferId] =
                l1Offset + aL1OneBuffer_ * (l1BufNum >> 1) + bL1OneBuffer_ * (bufferId >> 1);
        }
    }

    __aicore__ inline void UpdateParamsForNextProblem(const TupleShape& problemShape)
    {
        m_ = Get<GroupedMatmulFp16Recipe::MNK_M>(problemShape);
        n_ = Get<GroupedMatmulFp16Recipe::MNK_N>(problemShape);
        k_ = Get<GroupedMatmulFp16Recipe::MNK_K>(problemShape);
    }

    template <typename TensorA, typename TensorB, typename TensorC>
    __aicore__ inline void operator()(
        TensorA gmA, TensorB gmB, TensorC gmC, BlockShape singleShape)
    {
        Run(gmA, gmB, gmC, singleShape);
    }

private:
    // ---- 数据搬运: GM -> L1 ----
    // CopyAInL1: 将A矩阵的一个Tile从GM搬运到L1（NZlayout，用于Cube计算优化）
    template <typename TensorA>
    __aicore__ inline void CopyAInL1(
        TensorA gmA, uint64_t curM, uint64_t curGmAKL1, uint64_t aL1BufId, uint64_t kL1Offset)
    {
        auto copyGM2L1 = AscendC::Te::MakeCopy(AscendC::Te::CopyGM2L1{});
        auto layoutAL1 = MakeLayoutAL1{}(curM, curGmAKL1);
        auto tensorAL1 =
            AscendC::Te::MakeTensor(AscendC::Te::MakeL1memPtr<AType>(l1BufferAOffset_[aL1BufId]), layoutAL1);
        auto gmBlockA = gmA(AscendC::Te::MakeCoord(0, kL1Offset), AscendC::Te::MakeShape(curM, curGmAKL1));
        AscendC::Te::Copy(copyGM2L1, tensorAL1, gmBlockA);
    }

    template <typename TensorB>
    __aicore__ inline void CopyBInL1(
        TensorB gmB, uint64_t curN, uint64_t curGmBKL1, uint64_t bL1BufId, uint64_t kL1Offset)
    {
        auto copyGM2L1 = AscendC::Te::MakeCopy(AscendC::Te::CopyGM2L1{});
        auto layoutBL1 = MakeLayoutBL1{}(curGmBKL1, curN);
        auto tensorBL1 =
            AscendC::Te::MakeTensor(AscendC::Te::MakeL1memPtr<BType>(l1BufferBOffset_[bL1BufId]), layoutBL1);
        auto gmBlockB = gmB(AscendC::Te::MakeCoord(kL1Offset, 0), AscendC::Te::MakeShape(curGmBKL1, curN));
        AscendC::Te::Copy(copyGM2L1, tensorBL1, gmBlockB);
    }

    // ---- 核心计算: L1 -> L0 + MMAD Ping-Pong ----
    // Iterate: 沿K维度分步搬运，每次将baseK大小的数据从L1拷贝到L0，
    // 通过SetFlag/WaitFlag实现双缓冲Ping-Pong，L0数据ready后发起MMAD指令
    template <typename TensorL0C>
    __aicore__ inline void Iterate(
        TensorL0C tensorL0C, uint64_t curM, uint64_t curN, uint64_t curGmAKL1, uint64_t curGmBKL1,
        uint64_t aL1BufId, uint64_t bL1BufId, uint64_t absKStartA, uint64_t absKStartB,
        uint64_t kaL1Offset, uint64_t kbL1Offset)
    {
        auto tensorAL1 = AscendC::Te::MakeTensor(
            AscendC::Te::MakeL1memPtr<AType>(l1BufferAOffset_[aL1BufId]), MakeLayoutAL1{}(curM, curGmAKL1));
        auto tensorBL1 = AscendC::Te::MakeTensor(
            AscendC::Te::MakeL1memPtr<BType>(l1BufferBOffset_[bL1BufId]), MakeLayoutBL1{}(curGmBKL1, curN));

        auto CopyL12L0 = AscendC::Te::MakeCopy(AscendC::Te::CopyL12L0{});

        uint64_t minGmKL1 = Min(curGmAKL1, curGmBKL1);

        for (uint64_t kL0Offset = 0; kL0Offset < minGmKL1; kL0Offset += baseK_) {
            uint64_t curKL0 = (kL0Offset + baseK_ > minGmKL1) ? (minGmKL1 - kL0Offset) : baseK_;
            uint64_t l0Offset = HALF_L0_SIZE * (l0PingPong_ & 0x1);
            AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(l0PingPong_ & 0x1);

            auto tensorAL0 = AscendC::Te::MakeTensor(
                AscendC::Te::MakeL0AmemPtr<AType>(l0Offset), AscendC::Te::MakeNzLayout<AType>(curM, curKL0));
            auto tensorBlockAL1 = tensorAL1(
                AscendC::Te::MakeCoord(0, kaL1Offset + kL0Offset), AscendC::Te::MakeShape(curM, curKL0));
            AscendC::PipeBarrier<PIPE_ALL>();
            AscendC::Te::Copy(CopyL12L0, tensorAL0, tensorBlockAL1);

            auto tensorBL0 = AscendC::Te::MakeTensor(
                AscendC::Te::MakeL0BmemPtr<BType>(l0Offset), AscendC::Te::MakeZnLayout<BType>(curKL0, curN));
            auto tensorBlockBL1 = tensorBL1(
                AscendC::Te::MakeCoord(kbL1Offset + kL0Offset, 0), AscendC::Te::MakeShape(curKL0, curN));
            AscendC::Te::Copy(CopyL12L0, tensorBL0, tensorBlockBL1);

            AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(l0PingPong_ & 0x1);
            AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(l0PingPong_ & 0x1);

            // MMAD累积标记: 最后一段K使用FINAL_ACCUMULATION输出结果，其他段累加到L0C
            uint8_t mmadUnitFlag =
                (absKStartA + kaL1Offset + kL0Offset + baseK_ >= k_)
                    ? GroupedMatmulFp16Recipe::FINAL_ACCUMULATION
                    : GroupedMatmulFp16Recipe::NON_FINAL_ACCUMULATION;
            bool mmadCmatrixInitVal = (absKStartA + kaL1Offset + kL0Offset == 0);
            
            AscendC::MmadParams mmadPara;
            mmadPara.m = static_cast<uint16_t>(curM);
            mmadPara.n = static_cast<uint16_t>(curN);
            mmadPara.k = static_cast<uint16_t>(curKL0);
            mmadPara.cmatrixInitVal = mmadCmatrixInitVal;
            mmadPara.unitFlag = mmadUnitFlag;
            auto MadOp = AscendC::Te::MakeMad(AscendC::Te::MmadOperation{}, AscendC::Te::MmadTraitDefault{});
            AscendC::Te::Mad(MadOp, tensorL0C, tensorAL0, tensorBL0, mmadPara);
            AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(l0PingPong_ & 0x1);
            l0PingPong_++;
        }
    }

    // ---- Run: 将A_outer/B_outer次序下的K维度循环、L1搬运、L0 MMAD整合 ----
    // orderAL1BL1_决定外层循环是A(K)还是B(K)，根据kAL1/kBL1大小自动选择最优路径
    template <typename TensorA, typename TensorB, typename TensorC>
    __aicore__ inline void Run(
        TensorA gmA, TensorB gmB, TensorC gmC, const BlockShape& singleShape)
    {
        uint64_t curM = Get<GroupedMatmulFp16Recipe::MNK_M>(singleShape);
        uint64_t curN = Get<GroupedMatmulFp16Recipe::MNK_N>(singleShape);

        uint64_t l0cOffset = (l0cPingPong_ & 1) * HALF_L0C_SIZE;
        auto tensorL0C = AscendC::Te::MakeTensor(
            AscendC::Te::MakeL0CmemPtr<float>(l0cOffset), AscendC::Te::MakeL0CLayout(curM, curN));

        if (orderAL1BL1_) {
            for (uint64_t kOuter = 0; kOuter < k_; kOuter += kAL1_) {
                uint64_t aL1BufId = aL1LoopCnt_ & 1;
                uint64_t curGmAKL1 = (kOuter + kAL1_ > k_) ? (k_ - kOuter) : kAL1_;
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(aL1BufId);
                CopyAInL1(gmA, curM, curGmAKL1, aL1BufId, kOuter);
                for (uint64_t kInner = kOuter; kInner < Min(kOuter + kAL1_, k_); kInner += kBL1_) {
                    uint64_t bL1BufId = bL1LoopCnt_ & 1;
                    uint64_t curGmBKL1 = (kInner + kBL1_ > k_) ? (k_ - kInner) : kBL1_;
                    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(INPUT_BUFFER_MTE1_MTE2_BASE + bL1BufId);
                    CopyBInL1(gmB, curN, curGmBKL1, bL1BufId, kInner);
                    AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(bL1BufId);
                    AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(bL1BufId);
                    Iterate(tensorL0C, curM, curN, curGmAKL1, curGmBKL1, aL1BufId, bL1BufId,
                            kOuter, kInner, kInner - kOuter, 0);
                    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(INPUT_BUFFER_MTE1_MTE2_BASE + bL1BufId);
                    bL1LoopCnt_++;
                }
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(aL1BufId);
                aL1LoopCnt_++;
            }
        } else {
            for (uint64_t kOuter = 0; kOuter < k_; kOuter += kBL1_) {
                uint64_t bL1BufId = bL1LoopCnt_ & 1;
                uint64_t curGmBKL1 = (kOuter + kBL1_ > k_) ? (k_ - kOuter) : kBL1_;
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(bL1BufId);
                CopyBInL1(gmB, curN, curGmBKL1, bL1BufId, kOuter);
                for (uint64_t kInner = kOuter; kInner < Min(kOuter + kBL1_, k_); kInner += kAL1_) {
                    uint64_t aL1BufId = aL1LoopCnt_ & 1;
                    uint64_t curGmAKL1 = (kInner + kAL1_ > k_) ? (k_ - kInner) : kAL1_;
                    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(INPUT_BUFFER_MTE1_MTE2_BASE + aL1BufId);
                    CopyAInL1(gmA, curM, curGmAKL1, aL1BufId, kInner);
                    AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(aL1BufId);
                    AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(aL1BufId);
                    Iterate(tensorL0C, curM, curN, curGmAKL1, curGmBKL1, aL1BufId, bL1BufId,
                            kInner, kOuter, 0, kInner - kOuter);
                    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(INPUT_BUFFER_MTE1_MTE2_BASE + aL1BufId);
                    aL1LoopCnt_++;
                }
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(bL1BufId);
                bL1LoopCnt_++;
            }
        }

        auto CopyL0C2GM = AscendC::Te::MakeCopy(AscendC::Te::CopyL0C2GM{});
        AscendC::Te::Copy(CopyL0C2GM, gmC, tensorL0C,
                          AscendC::Te::FixpipeParams{GroupedMatmulFp16Recipe::FINAL_ACCUMULATION});
        if (enableL0cPingPong_) {
            l0cPingPong_++;
        }
    }

    // ---- 内部状态 ----
    // L1双缓冲池: A和B各2份buffer，通过bufferId切换实现Ping-Pong
    // L0双缓冲: l0PingPong_控制A/B Buffer切换，l0cPingPong_控制C Buffer切换
    uint64_t aL1OneBuffer_ = 0UL;
    uint64_t bL1OneBuffer_ = 0UL;
    uint64_t l1BufferAOffset_[2] = {0UL};
    uint64_t l1BufferBOffset_[2] = {0UL};
    uint64_t m_{0};
    uint64_t n_{0};
    uint64_t k_{0};
    uint64_t kAL1_{1};
    uint64_t kBL1_{1};
    uint64_t baseM_{16};
    uint64_t baseN_{16};
    uint64_t baseK_{16};
    uint64_t aL1LoopCnt_{0};
    uint64_t bL1LoopCnt_{0};
    uint64_t l0PingPong_{0};
    uint64_t l0cPingPong_{0};
    bool orderAL1BL1_{false};
    bool enableL0cPingPong_{false};
    AscendC::LocalTensor<bfloat16_t> l0aLocal_{AscendC::TPosition::A2, 0, L0A_SIZE};
    AscendC::LocalTensor<bfloat16_t> l0bLocal_{AscendC::TPosition::B2, 0, L0B_SIZE};
    AscendC::LocalTensor<float> c1Local_{AscendC::TPosition::CO1, 0, L0C_SIZE};
    AscendC::LocalTensor<bfloat16_t> aL1Local_{AscendC::TPosition::A1, 0, L1_SIZE};
    AscendC::LocalTensor<bfloat16_t> bL1Local_{AscendC::TPosition::A1, 0, L1_SIZE};
};
} // namespace Block

#endif


`BlockMmadFp16` 是 Matmul 的最小计算单元，核心特性：

- **Ping-Pong 双缓冲**：L1 Buffer 和 L0C Buffer 均采用双缓冲，实现数据搬运与计算的 Overlap
- **数据流**：GM → L1（CopyAInL1/CopyBInL1）→ L0A/L0B（硬件自动）→ MMAD 计算 → L0C
- **Iterate 循环**：沿 K 维度循环，每次处理 `kAL1`/`kBL1` 长度
- **构造/析构中的 SetFlag/WaitFlag**：保证硬件事件同步
- 仅支持非转置的 A 矩阵和 B 矩阵（`static_assert(!transA)` 和 `static_assert(!transB)`）

### **5.4 Grouped Matmul Kernel**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/kernel/grouped_matmul_fp16_kernel_split_m.h
#ifndef GROUPED_MATMUL_FP16_KERNEL_SPLIT_M_H
#define GROUPED_MATMUL_FP16_KERNEL_SPLIT_M_H

#include "kernel_basic_intf.h"
#include "experimental/tensor_api/tensor.h"
#include "../block/block_mmad_fp16_split_m.h"
#include "../block/block_scheduler_fp16_split_m.h"
#include "../block/block_scheduler_utils.h"
#include "../tiling/grouped_matmul_fp16_tiling_data.h"
#include "../utils/grouped_matmul_fp16_constant.h"

namespace Kernel {

// ============================================================
// KernelGmmFp16: Split-M Grouped Matmul Kernel
// 将M维度按splitM分组，每组通过BlockScheduler分配Tile给各Block
// 每组内循环: GetTileIdx -> GetBlockShape -> mmadOp_(tileA, tileB, tileC)
// ============================================================
template <class ProblemShape, class BlockMmad, class BlockScheduler>
class KernelGmmFp16 {
public:
    static constexpr bool transA = BlockMmad::transA;
    static constexpr bool transB = BlockMmad::transB;
    static_assert(!transA, "KernelGmmFp16 only supports non-transposed A.");
    static_assert(!transB, "KernelGmmFp16 only supports non-transposed B (ND).");
    using AType = typename BlockMmad::AType;
    using BType = typename BlockMmad::BType;
    using CType = typename BlockMmad::CType;
    using TupleShape = AscendC::Shape<int64_t, int64_t, int64_t>;
    using BlockShape = AscendC::Shape<int64_t, int64_t, int64_t, int64_t>;
    using BlockCoord = AscendC::Coord<int64_t, int64_t, int64_t, int64_t>;
    using BlockOffset = AscendC::Shape<int64_t, int64_t, int64_t>;
    using BlockMmadShape = typename BlockMmad::BlockShape;
    using MakeLayoutA = AscendC::Te::NDLayoutFormat<AType>;
    using MakeLayoutB = AscendC::Te::NDLayoutFormat<BType>;

    struct Params {
        ProblemShape problemShape;
        typename BlockMmad::Params mmadParams;
        const GroupedMatmulFp16TilingData* gmmParams{nullptr};
        uint32_t splitId{0};
        uint32_t chunkM{0};
        Params() = default;
    };

    __aicore__ inline void operator()(const Params& params)
    {
        Run(params);
    }

private:
    __aicore__ inline void ResetGmAddr(const Params& params)
    {
        xGmAddr_ = reinterpret_cast<__gm__ AType*>(params.mmadParams.aGmAddr);
        wGmAddr_ = reinterpret_cast<__gm__ BType*>(params.mmadParams.bGmAddr);
        yGmAddr_ = reinterpret_cast<__gm__ CType*>(params.mmadParams.cGmAddr);
    }

    // Run: 分组处理流程
    // 1. ResetGmAddr: 设置GM地址
    // 2. Init: 初始化Tiling参数和MMAD算子
    // 3. 遍历每组: UpdateOffset(计算GM偏移) -> SetMNK -> BaseMBalance -> ProcessSingleGroup
    __aicore__ inline void Run(const Params& params)
    {
        ResetGmAddr(params);
        Init(params);
        using SchedulerOp = BlockScheduler;
        SchedulerOp bs(static_cast<int32_t>(params.gmmParams->baseM), static_cast<int32_t>(params.gmmParams->baseN),
                       static_cast<int32_t>(params.gmmParams->baseK), params.gmmParams->usedCoreNum);
        for (uint32_t groupIdx = 0; groupIdx < groupNum_; ++groupIdx) {
            UpdateOffset(groupIdx);
            SetMNK(groupIdx);
            if (AscendC::Std::get<MNK_M>(problemShape_) <= 0 || AscendC::Std::get<MNK_K>(problemShape_) <= 0) {
                continue;
            }
            mmadOp_.UpdateParamsForNextProblem(problemShape_);
            BaseMBalance(bs, AscendC::Std::get<MNK_M>(problemShape_), params.gmmParams->baseM);
            typename SchedulerOp::TupleShape bsProblemShape{
                AscendC::Std::get<MNK_M>(problemShape_), AscendC::Std::get<MNK_N>(problemShape_),
                AscendC::Std::get<MNK_K>(problemShape_), 1L};
            bs.UpdateNextProblem(bsProblemShape);
            if (IsLastGroupAndNeedSplit(bs, groupIdx)) {
                bs.UpdateTailTile();
            }
            ProcessSingleGroup(params, bs);
        }
    }

    __aicore__ inline void Init(const Params& params)
    {
        groupNum_ = params.gmmParams->groupNum;
        curBaseM_ = params.gmmParams->baseM;
        splitId_ = params.splitId;
        chunkM_ = params.chunkM;
        k_ = params.gmmParams->k;
        n_ = params.gmmParams->n;
        splitM_ = params.gmmParams->maxM;
        baseOffset_ = {0, 0, 0};
        AscendC::Std::get<MNK_M>(problemShape_) = params.gmmParams->maxM;
        AscendC::Std::get<MNK_N>(problemShape_) = params.gmmParams->n;
        AscendC::Std::get<MNK_K>(problemShape_) = params.gmmParams->k;
        TupleShape l0Shape{static_cast<int64_t>(params.gmmParams->baseM), static_cast<int64_t>(params.gmmParams->baseN),
                           static_cast<int64_t>(params.gmmParams->baseK)};
        typename BlockMmad::L1Params l1Params{static_cast<uint64_t>(params.gmmParams->kAL1),
                                              static_cast<uint64_t>(params.gmmParams->kBL1)};
        mmadOp_.Init(problemShape_, l0Shape, l1Params, params.gmmParams->dbL0C == GroupedMatmulFp16Recipe::DOUBLE_BUFFER);
    }

    template <class SchedulerOp>
    __aicore__ inline void BaseMBalance(SchedulerOp& bs, int64_t m, int64_t baseM)
    {
        int64_t mCnt = CeilDiv(m, baseM);
        curBaseM_ = CeilAlign(CeilDiv(m, mCnt), AscendC::BLOCK_CUBE);
        bs.UpdateBaseM(curBaseM_);
    }

    template <class SchedulerOp>
    __aicore__ inline bool IsLastGroupAndNeedSplit(const SchedulerOp& bs, uint32_t groupIdx) const
    {
        return false;
    }

    __aicore__ inline void SetMNK(uint32_t groupIdx)
    {
        AscendC::Std::get<MNK_M>(problemShape_) = splitM_;
    }

    __aicore__ inline void UpdateOffset(uint32_t groupIdx)
    {
        if (groupIdx == 0) {
            AscendC::Std::get<0>(baseOffset_) = splitM_ * k_ * splitId_;
            AscendC::Std::get<1>(baseOffset_) = 0;
            AscendC::Std::get<2>(baseOffset_) = splitM_ * n_ * splitId_;
            return;
        }
        AscendC::Std::get<0>(baseOffset_) += chunkM_ * k_;
        AscendC::Std::get<1>(baseOffset_) = 0;
        AscendC::Std::get<2>(baseOffset_) += chunkM_ * n_;
    }

    // ProcessSingleGroup: 处理单个Group的完整Matmul
    // 通过BlockScheduler获取Tile分配 -> 计算GM偏移 -> 调用mmadOp_执行MMAD
    template <class SchedulerOp>
    __aicore__ inline void ProcessSingleGroup(const Params& params, SchedulerOp& bs)
    {
        int64_t groupM = AscendC::Std::get<MNK_M>(problemShape_);
        int64_t groupN = AscendC::Std::get<MNK_N>(problemShape_);
        int64_t groupK = AscendC::Std::get<MNK_K>(problemShape_);
        __gm__ AType* groupAPtr = xGmAddr_ + AscendC::Std::get<0>(baseOffset_);
        __gm__ BType* groupBPtr = wGmAddr_ + AscendC::Std::get<1>(baseOffset_);
        __gm__ CType* groupCPtr = yGmAddr_ + AscendC::Std::get<2>(baseOffset_);
        auto layoutA = MakeLayoutA{}(groupM, groupK);
        auto layoutB = MakeLayoutB{}(groupK, groupN);
        auto groupA = AscendC::Te::MakeTensor(AscendC::Te::MakeGMmemPtr(groupAPtr), layoutA);
        auto groupB = AscendC::Te::MakeTensor(AscendC::Te::MakeGMmemPtr(groupBPtr), layoutB);
        auto groupC = AscendC::Te::MakeTensor(
            AscendC::Te::MakeGMmemPtr(groupCPtr),
            AscendC::Te::MakeNDLayout<CType>(groupM, groupN));
        BlockCoord tileIdx;
        while (bs.GetTileIdx(tileIdx)) {
            BlockShape singleShape = bs.GetBlockShape(tileIdx);
            if (AscendC::Std::get<MNK_M>(singleShape) <= 0 || AscendC::Std::get<MNK_N>(singleShape) <= 0) {
                return;
            }
            int64_t mOffset = AscendC::Std::get<0>(tileIdx) * static_cast<int64_t>(curBaseM_) +
                              AscendC::Std::get<2>(singleShape);
            int64_t nOffset = AscendC::Std::get<1>(tileIdx) * static_cast<int64_t>(params.gmmParams->baseN) +
                              AscendC::Std::get<3>(singleShape);
            int64_t tileM = AscendC::Std::get<MNK_M>(singleShape);
            int64_t tileN = AscendC::Std::get<MNK_N>(singleShape);
            auto gmBlockA = groupA(
                AscendC::Te::MakeCoord(mOffset, 0),
                AscendC::Te::MakeShape(tileM, groupK));
            auto gmBlockB = groupB(
                AscendC::Te::MakeCoord(0, nOffset),
                AscendC::Te::MakeShape(groupK, tileN));
            auto gmBlockC = groupC(
                AscendC::Te::MakeCoord(mOffset, nOffset),
                AscendC::Te::MakeShape(tileM, tileN));
            BlockMmadShape mmadShape{
                tileM, tileN, static_cast<int64_t>(groupK)};
            mmadOp_(gmBlockA, gmBlockB, gmBlockC, mmadShape);
        }
    }

private:
    BlockMmad mmadOp_;
    TupleShape problemShape_{};
    BlockOffset baseOffset_{0, 0, 0};
    __gm__ AType* xGmAddr_{nullptr};
    __gm__ BType* wGmAddr_{nullptr};
    __gm__ CType* yGmAddr_{nullptr};
    uint32_t groupNum_{0};
    uint32_t curBaseM_{0};
    uint32_t splitId_{0};
    uint32_t chunkM_{0};
    uint32_t k_{0};
    uint32_t n_{0};
    uint32_t splitM_{0};
};

template <class ProblemShape, class BlockMmad, class BlockScheduler>
using GroupedMatmulFp16KernelSplitM = KernelGmmFp16<ProblemShape, BlockMmad, BlockScheduler>;

} // namespace Kernel

#endif


`KernelGmmFp16` 作为 Matmul 模块的顶层编排类：

- **Split-M 分组**：将 M 维度按 `splitM` 分组，每组通过 BlockScheduler 分配 Tile
- **ProcessSingleGroup**：处理单个 group 的完整 Matmul 流程（Init → ResetGmAddr → BlockScheduler 循环 → BlockMmad 计算）
- **BaseMBalance**：平衡每个 AIC 核的 M 维度负载
- **UpdateOffset**：根据 groupIdx 计算 GM 地址偏移

Split-M 数据布局示意（4 Rank, M=1024, splitMCnt=2）：
```
Rank0: M行[0,255]   → Split0: M[0,127],   Split1: M[128,255]
Rank1: M行[256,511] → Split0: M[256,383], Split1: M[384,511]
Rank2: M行[512,767] → Split0: M[512,639], Split1: M[640,767]
Rank3: M行[768,1023]→ Split0: M[768,895], Split1: M[896,1023]
```

---
## **6. MTE 通信模块（ReduceScatter）**

MTE 通信模块运行在 AIV 核上，负责 Win Area 管理、状态同步和 ReduceScatter 数据收集。

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/kernel/mte_comm.h
#ifndef MTE_COMM_REDUCE_SCATTER_H
#define MTE_COMM_REDUCE_SCATTER_H

#include "kernel_operator.h"
#include "../utils/reduce_scatter_utils.h"

namespace MatmulReduceScatterImpl {

using namespace AscendC;

constexpr static uint32_t UB_ALIGN_BYTES = 32U;
constexpr static uint32_t FLOAT_ALIGN_NUM = 8U;
constexpr static uint32_t BUFFER_NUM = 2U;
constexpr static uint32_t MAX_PER_BLOCK_NUM = 1024U * 15U;
constexpr static uint64_t STATE_WIN_SIZE = 2UL * 1024UL * 1024UL;
constexpr static uint64_t SINGLE_STATE_REGION_SIZE = 512UL * 1024UL;

constexpr static int64_t MAX_RANK_NUM = 64;
// UB预算分配: 状态区(Status同步) + 数据区(xQueue_ + computeQueue_)各占1/4 UB

struct HcclA5OpResParam {
    uint64_t workSpace;
    uint64_t workSpaceSize;
    uint32_t rankId;
    uint32_t rankDim;
    uint64_t winSize;
    uint64_t windowsIn[MAX_RANK_NUM];
    uint64_t windowsOut[MAX_RANK_NUM];
    uint64_t xnAddr;
    uint64_t ckeAddr;
    uint64_t msAddr;
    uint64_t msSize;
};
// HcclA5OpResParam: MC2通信资源描述
// - windowsIn[rankDim]: 各rank的Win Area输入窗口基地址
// - windowsOut[rankDim]: 各rank的Win Area输出窗口基地址
// - winSize: 单个Win Area的大小(Rank0写Matmul结果，Rank1..读取)
// - rankId/rankDim: 当前rankID和总rank数

template<typename DataType>
class MTECommunication {
public:
    __aicore__ inline MTECommunication() {};
    __aicore__ inline void InitHcclContextByAddr(GM_ADDR mc2Context);
    __aicore__ inline void ComputeBlockParams();
    __aicore__ inline void InitParams();
    __aicore__ inline void InitGMTensor(GM_ADDR y, uint64_t ySize, uint64_t chunkSize, uint64_t winSpaceGmSize);
    __aicore__ inline void InitBuffer(TPipe *tPipe);
    __aicore__ inline void SetBlockSize(uint32_t elementsPerBlock, uint64_t aivNum, uint64_t lastBlockNum);
    __aicore__ inline void WriteStatusToWin(uint32_t splitId);
    __aicore__ inline void ReadStatus(uint32_t splitId);
    __aicore__ inline void ComputeTailAivId(uint64_t totalAivCount);
    // ReduceScatter核心函数：从各rank Win Area读取数据并做ReduceSum
    __aicore__ inline void ReduceGatherFromWin(uint32_t splitId);
    __aicore__ inline GM_ADDR GetWinAddrGm(uint32_t rankId, uint64_t offset = 0);
    __aicore__ inline GM_ADDR GetWinDataAddrGm(uint32_t rankId, uint32_t winFlag);
    __aicore__ inline GM_ADDR GetWinStatusAddrGm(uint32_t rankId, uint32_t winFlag);
    __aicore__ inline GM_ADDR GetLocalWinOutAddr();

    __gm__ HcclA5OpResParam *hcclContext_;
    uint32_t aivId_{0};
    uint64_t aivNum_{0};
    uint32_t round_{0};
    uint32_t tailBlockNums_{0};
    uint32_t assignedBlockNums_{0};
    uint64_t xOffset_{0};
    uint64_t lastAivId_{0};
    uint32_t winBufferFlags_{0};
    uint32_t xNumPerBlock_{0};
    uint64_t tailXNums_{0};
    uint64_t xNums_{0};
    uint64_t chunkSize_{0};
    uint64_t totalLen_{0};
    uint32_t m_{0};
    uint32_t n_{0};
    uint32_t splitM_{0};
    GlobalTensor<DataType> outputTensor_;
    GlobalTensor<DataType> localWinOutTensor_;

private:
    // ReduceScatter: 从远端Win读取数据块并累加
    __aicore__ inline void ReduceDataBlockFromWin(uint32_t remoteRankId, uint64_t readOffset, 
                                                   uint64_t curOutOffset, uint32_t count,
                                                   LocalTensor<DataType>& accumTensor, bool isFirst);

    TQueBind<QuePosition::VECIN, QuePosition::VECOUT, 1> xQueue_;
    TQueBind<QuePosition::VECIN, QuePosition::VECOUT, 1> computeQueue_;
    TBuf<> writeStateBuf_;
    TBuf<> readStateBuf_;
    TBuf<> stateResetBuf_;

    LocalTensor<DataType> xTmpTensor_;
    LocalTensor<DataType> accumTensor_;
    LocalTensor<float> stateResetTensor_;
};

template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::InitHcclContextByAddr(GM_ADDR mc2Context)
{
    hcclContext_ = (__gm__ HcclA5OpResParam*)(mc2Context);
}

template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::ComputeBlockParams()
{
    // UB预算: 状态区 = 固定开销(2个32B对齐块) + rankDim*2个32B对齐块
    uint64_t stateBufSize = UB_ALIGN_BYTES * 2 + hcclContext_->rankDim * UB_ALIGN_BYTES * 2;
    // 剩余UB空间 = 总UB - 状态区，用于数据搬运队列
    uint64_t availableUbForData = TOTAL_UB_SIZE - stateBufSize;
    // ReduceScatter需要两份UB队列空间：xQueue_和computeQueue_，每个队列BUFFER_NUM=2，总共4份
    uint64_t maxPerBlockBytes = FloorAlign(availableUbForData / 4, UB_ALIGN_BYTES);
    uint64_t maxPerBlockNum = maxPerBlockBytes / sizeof(DataType);
    xNumPerBlock_ = static_cast<uint32_t>(MIN(MAX_PER_BLOCK_NUM, maxPerBlockNum));
}

template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::InitParams()
{
    aivId_ = GetBlockIdx();
    assignedBlockNums_ = aivId_ < tailBlockNums_ ? round_ + 1 : round_;
    uint64_t blockIdx = aivId_ * round_ + (aivId_ < tailBlockNums_ ? aivId_ : tailBlockNums_);
    xOffset_ = blockIdx * xNumPerBlock_;
}

template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::InitGMTensor(GM_ADDR y, uint64_t ySize, uint64_t chunkSize, uint64_t winSpaceGmSize)
{
    outputTensor_.SetGlobalBuffer((__gm__ DataType*)y);

    // 默认使用winBufferFlags=0（不从Win Area读取未初始化的状态）
    winBufferFlags_ = 0;

    // 本卡Win Area数据区（Matmul结果写入位置）
    GM_ADDR localWinDataGm = GetWinDataAddrGm(hcclContext_->rankId, winBufferFlags_);
    localWinOutTensor_.SetGlobalBuffer((__gm__ DataType*)localWinDataGm);
}

template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::InitBuffer(TPipe *tPipe)
{
    // ReduceScatter需要额外的computeQueue用于Reduce计算
    tPipe->InitBuffer(xQueue_, BUFFER_NUM, xNumPerBlock_ * sizeof(DataType));
    tPipe->InitBuffer(computeQueue_, BUFFER_NUM, xNumPerBlock_ * sizeof(DataType));
    tPipe->InitBuffer(writeStateBuf_, UB_ALIGN_BYTES);
    tPipe->InitBuffer(readStateBuf_, hcclContext_->rankDim * UB_ALIGN_BYTES);
    tPipe->InitBuffer(stateResetBuf_, hcclContext_->rankDim * UB_ALIGN_BYTES);

    stateResetTensor_ = stateResetBuf_.Get<float>();
    Duplicate<float>(stateResetTensor_, (float)0.0, static_cast<uint32_t>(hcclContext_->rankDim * FLOAT_ALIGN_NUM));
}

template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::SetBlockSize(uint32_t elementsPerBlock, uint64_t aivNum, uint64_t lastBlockNum)
{
    xNumPerBlock_ = elementsPerBlock;
    aivNum_ = aivNum;
    tailXNums_ = lastBlockNum;
}

template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::ComputeTailAivId(uint64_t totalAivCount)
{
    if (round_ == 0) {
        lastAivId_ = tailBlockNums_ - 1;
    } else {
        lastAivId_ = totalAivCount - 1;
    }
}

template<typename DataType>
// ============================================================
// WriteStatusToWin: AIV核向所有Rank的状态区写入本Rank完成标记
// Status区布局(每rank):
//   [0..MAX_AIV_NUM*rankDim*FLOAT_ALIGN_NUM)  = Split0的Status区
//   [MAX_AIV_NUM*rankDim*FLOAT_ALIGN_NUM .. ] = Split1的Status区
// 每个Cell: 写入1.0f(已完成标记)，ReadStatus通过Sum求和确认所有rank完成
// ============================================================
__aicore__ inline void MTECommunication<DataType>::WriteStatusToWin(uint32_t splitId)
{
    
    constexpr uint32_t MAX_AIV_NUM = 56U;
    uint32_t statusCoreBase = aivId_ * hcclContext_->rankDim;
    uint64_t splitOffset = splitId * MAX_AIV_NUM * hcclContext_->rankDim * FLOAT_ALIGN_NUM;

    for (uint32_t targetRank = 0; targetRank < hcclContext_->rankDim; ++targetRank) {
        // 写入目标rank的状态区
        GM_ADDR targetWinStateGM = GetWinStatusAddrGm(targetRank, winBufferFlags_);
        GlobalTensor<float> stateGMTensor;
        stateGMTensor.SetGlobalBuffer((__gm__ float*)targetWinStateGM);

        // 写入位置 = splitOffset + (statusCoreBase + rankId) * FLOAT_ALIGN_NUM
        uint64_t statusWriteOffset = splitOffset + (statusCoreBase + hcclContext_->rankId) * FLOAT_ALIGN_NUM;

        // ★Step 1: 先清零目标槽位（避免残留值叠加）
        SyncFunc<AscendC::HardEvent::S_MTE3>();
        DataCopyExtParams stateCopyOutParams{1, static_cast<uint32_t>(FLOAT_ALIGN_NUM * sizeof(float)), 0, 0, 0};
        DataCopyPad(stateGMTensor[statusWriteOffset], stateResetTensor_, stateCopyOutParams);
        SyncFunc<AscendC::HardEvent::MTE3_S>();

        // ★Step 2: 准备状态tensor并写入1.0
        LocalTensor<float> statusTensor = writeStateBuf_.Get<float>();
        DataCopy<float>(statusTensor, stateResetTensor_, FLOAT_ALIGN_NUM);
        SyncFunc<AscendC::HardEvent::MTE2_S>();
        statusTensor.SetValue(0, (float)1.0);

        // ★Step 3: 写入状态值
        SyncFunc<AscendC::HardEvent::S_MTE3>();
        DataCopyPad(stateGMTensor[statusWriteOffset], statusTensor, stateCopyOutParams);
        SyncFunc<AscendC::HardEvent::MTE3_S>();
    }
}

template<typename DataType>
// ============================================================
// ReadStatus: AIV核轮询本Rank状态区，等待所有Rank完成Matmul
// 轮询: 读取Status区 -> Sum求和 -> 判断 == rankDim? -> 清零准备下一步
// ============================================================
__aicore__ inline void MTECommunication<DataType>::ReadStatus(uint32_t splitId)
{
    
    constexpr uint32_t MAX_AIV_NUM = 56U;
    uint64_t splitOffset = splitId * MAX_AIV_NUM * hcclContext_->rankDim * FLOAT_ALIGN_NUM;
    
    // 从本rank状态区读取
    GM_ADDR stateGM = GetWinStatusAddrGm(hcclContext_->rankId, winBufferFlags_);
    GlobalTensor<float> selfStatusWinTensor;
    uint32_t statusCoreOffset = aivId_ * hcclContext_->rankDim * FLOAT_ALIGN_NUM;
    selfStatusWinTensor.SetGlobalBuffer((__gm__ float*)stateGM);
    
    // 实际读取偏移 = splitOffset + statusCoreOffset
    uint64_t readOffset = splitOffset + statusCoreOffset;
    
    LocalTensor<float> statusTensor = readStateBuf_.Get<float>();
    float flag = 0;
    uint32_t statusCnt = hcclContext_->rankDim * FLOAT_ALIGN_NUM;
    SumParams sumParams{1, statusCnt, statusCnt};
    float minTarget = hcclContext_->rankDim - 0.5f;
    float maxTarget = hcclContext_->rankDim + 0.5f;
    
    // 轮询等待所有rank完成写入
    while ((flag < minTarget) || (flag > maxTarget)) {
        SyncFunc<AscendC::HardEvent::S_MTE2>();
        DataCopyExtParams statusCopyInParams{1, static_cast<uint32_t>(statusCnt * sizeof(float)), 0, 0, 0};
        DataCopyPadExtParams<float> statusPadParams{false, 0, 0, 0};
        DataCopyPad(statusTensor, selfStatusWinTensor[readOffset], statusCopyInParams, statusPadParams);
        SyncFunc<AscendC::HardEvent::MTE2_V>();
        Sum(statusTensor, statusTensor, sumParams);
        SyncFunc<AscendC::HardEvent::V_S>();
        flag = statusTensor.GetValue(0);
    }
    
    // 清零状态区，为下一轮通信做准备
    SyncFunc<AscendC::HardEvent::S_MTE3>();
    DataCopyExtParams statusCopyOutParams{1, static_cast<uint32_t>(statusCnt * sizeof(float)), 0, 0, 0};
    DataCopyPad(selfStatusWinTensor[readOffset], stateResetTensor_, statusCopyOutParams);
}

// ============================================================
// ReduceDataBlockFromWin: 从remoteRankId的Win Area读取一个数据块
// isFirst=true: 直接Copy到accumTensor (第一个rank的数据)
// isFirst=false: 与accumTensor做Add累加 (后续rank的贡献)
// ============================================================
template<typename DataType>
__aicore__ inline void MTECommunication<DataType>::ReduceDataBlockFromWin(
    uint32_t remoteRankId, uint64_t readOffset, uint64_t curOutOffset, uint32_t count,
    LocalTensor<DataType>& accumTensor, bool isFirst)
{
    // 与AllGather保持一致，使用GetWinDataAddrGm + winBufferFlags_
    GM_ADDR remoteWinDataGm = GetWinDataAddrGm(remoteRankId, winBufferFlags_);
    GlobalTensor<DataType> remoteWinXTensor;
    remoteWinXTensor.SetGlobalBuffer((__gm__ DataType*)remoteWinDataGm);

    xTmpTensor_ = xQueue_.AllocTensor<DataType>();
    DataCopyExtParams copyInParams{1, static_cast<uint32_t>(count * sizeof(DataType)), 0, 0, 0};
    DataCopyPadExtParams<DataType> padParams{false, 0, 0, 0};
    DataCopyPad(xTmpTensor_, remoteWinXTensor[readOffset], copyInParams, padParams);
    xQueue_.EnQue(xTmpTensor_);
    SyncFunc<AscendC::HardEvent::MTE2_V>();  // 等待数据拷贝完成
    xTmpTensor_ = xQueue_.DeQue<DataType>();

    if (isFirst) {
        // 第一个rank的数据直接拷贝到累加缓冲区
        // UB内部拷贝使用DataCopy（Duplicate用于scalar复制）
        DataCopy(accumTensor, xTmpTensor_, count);
        PipeBarrier<PIPE_V>();  // 确保拷贝完成，防止数据覆盖
    } else {
        // 后续rank的数据做Add累加
        Add(accumTensor, accumTensor, xTmpTensor_, count);
        PipeBarrier<PIPE_V>();  // 确保Add完成，防止数据覆盖
    }
    
    xQueue_.FreeTensor(xTmpTensor_);
}

template<typename DataType>
// ============================================================
// ReduceGatherFromWin: ReduceScatter核心
// 循环遍历本Rank负责的所有数据块
// 每个数据块: 遍历所有srcRank -> ReduceDataBlockFromWin(读+累加) -> Copy到GM输出
// ============================================================
__aicore__ inline void MTECommunication<DataType>::ReduceGatherFromWin(uint32_t splitId)
{
    uint64_t chunkOffset = hcclContext_->rankId * chunkSize_;
    uint64_t splitOffset = splitId * static_cast<uint64_t>(splitM_) * static_cast<uint64_t>(n_);

    for (uint64_t curBlock = 0; curBlock < assignedBlockNums_; ++curBlock) {
        uint64_t curReadOffset = chunkOffset + splitOffset + xOffset_ + curBlock * xNumPerBlock_;
        uint64_t curOutOffset = splitOffset + xOffset_ + curBlock * xNumPerBlock_;
        uint32_t copyBlockNum = xNumPerBlock_;

        if ((aivId_ == lastAivId_) && (curBlock == assignedBlockNums_ - 1)) {
            copyBlockNum = tailXNums_;
        }

        accumTensor_ = computeQueue_.AllocTensor<DataType>();

        for (uint32_t srcRank = 0; srcRank < hcclContext_->rankDim; ++srcRank) {
            ReduceDataBlockFromWin(srcRank, curReadOffset, curOutOffset, copyBlockNum, accumTensor_, srcRank == 0);
        }

        SyncFunc<AscendC::HardEvent::V_MTE3>();

        DataCopyExtParams copyOutParams{1, static_cast<uint32_t>(copyBlockNum * sizeof(DataType)), 0, 0, 0};
        DataCopyPad(outputTensor_[curOutOffset], accumTensor_, copyOutParams);
        
        computeQueue_.FreeTensor(accumTensor_);
    }
    
    AscendC::PipeBarrier<PIPE_ALL>();
}

template<typename DataType>
__aicore__ inline GM_ADDR MTECommunication<DataType>::GetWinAddrGm(uint32_t rankId, uint64_t offset)
{
    return (GM_ADDR)(hcclContext_->windowsIn[rankId] + offset);
}

template<typename DataType>
__aicore__ inline GM_ADDR MTECommunication<DataType>::GetWinDataAddrGm(uint32_t rankId, uint32_t winFlag)
{
    if (winFlag == 0U) {
        return GetWinAddrGm(rankId, STATE_WIN_SIZE);
    } else {
        return (GM_ADDR)(hcclContext_->windowsOut[rankId]);
    }
}

template<typename DataType>
__aicore__ inline GM_ADDR MTECommunication<DataType>::GetWinStatusAddrGm(uint32_t rankId, uint32_t winFlag)
{
    if (winFlag == 0U) {
        return GetWinAddrGm(rankId);
    } else {
        return GetWinAddrGm(rankId, SINGLE_STATE_REGION_SIZE);
    }
}

template<typename DataType>
__aicore__ inline GM_ADDR MTECommunication<DataType>::GetLocalWinOutAddr()
{
    return GetWinDataAddrGm(hcclContext_->rankId, winBufferFlags_);
}

}

#endif

`MTECommunication` 类的核心成员：

| 成员/函数 | 功能 |
|-----------|------|
| `InitHcclContextByAddr(mc2Context)` | 获取 Device 端通信上下文指针 |
| `ComputeBlockParams()` | 根据可用 UB 大小计算每 Block 处理的元素数 |
| `InitGMTensor(y, ySize, chunkSize, ...)` | 设置输出和 Win Area 的 GM Tensor |
| `WriteStatusToWin(splitId)` | 向所有 Rank 的 Status 区写入完成标记 |
| `ReadStatus(splitId)` | 轮询等待所有 Rank 完成 |
| `ReduceGatherFromWin(splitId)` | **核心**：从各 Rank Win Area 读取 → 累加求和 → 输出 |
| `ReduceDataBlockFromWin(...)` | 从单个 Rank 读取一个数据块并累加到 accumTensor |
| `GetWinAddrGm(rankId)` | 获取指定 Rank 的 Win Area 基地址 |

**Status 同步机制**：AIV 核向 Win Area 的 Status 区写入 `1.0f` 作为完成标记，然后轮询检查所有 Rank 的 Status 累积值是否达到 `rankDim`，确保所有设备 Matmul 完成后再启动 ReduceScatter。

**UB Buffer 布局（AIV 核上）**：
- `xQueue_`：VECIN/VECOUT 绑定队列，接收从 Win Area 读取的数据
- `computeQueue_`：VECIN/VECOUT 绑定队列，用于累加计算
- `stateResetBuf_/writeStateBuf_/readStateBuf_`：Status 读写 Buffer
- `accumTensor_`：累加 Tensor（float32），存储 ReduceScatter 中间结果

---
## **7. 顶层 Kernel 流程编排**

`MatmulReduceScatterKernel` 是算子的顶层调度器，负责 Matmul 和 ReduceScatter 的 Split 级流水编排。

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/include/kernel/matmul_reduce_scatter_kernel.h
#ifndef MATMUL_REDUCE_SCATTER_KERNEL_H
#define MATMUL_REDUCE_SCATTER_KERNEL_H

#include "kernel_operator.h"
#include "mte_comm.h"
#include "../utils/reduce_scatter_utils.h"
#include "../tiling/matmul_reduce_scatter_tiling.h"
#include "../kernel/grouped_matmul_fp16_kernel_split_m.h"
#include "../block/block_mmad_fp16_split_m.h"
#include "../block/block_scheduler_fp16_split_m.h"
#include "../block/block_scheduler_utils.h"
#include "../tiling/grouped_matmul_fp16_tiling_data.h"
#include "../utils/grouped_matmul_fp16_constant.h"

namespace MatmulReduceScatterImpl {

using namespace AscendC;

// ============================================================
// MatmulReduceScatterKernel: MC2融合算子顶层调度器
// 核类型: AIC(AI Control)执行Matmul, AIV(AI Vector)执行ReduceScatter
// 通过ASCEND_IS_AIV/ASCEND_IS_AIC宏区分核角色
// ============================================================
template<typename DataType>
class MatmulReduceScatterKernel {
public:
    __aicore__ inline MatmulReduceScatterKernel() {};
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR weight, GM_ADDR y,
                                 GM_ADDR mc2Context, GM_ADDR tilingGM, TPipe *tPipe);
    __aicore__ inline void Process();

private:
    __aicore__ inline void ExecuteMatmul(uint32_t splitId);
    __aicore__ inline void WriteMatmulStatus();
    __aicore__ inline void ReadMatmulStatus();
    __aicore__ inline void ExecuteReduceScatter(uint32_t splitId);

    __gm__ MatmulReduceScatterTilingData* tilingData_;
    MTECommunication<DataType> mteComm_;

    uint64_t ySize_{0};
    uint64_t chunkSize_{0};
    uint64_t yNums_{0};
    uint64_t chunkNums_{0};
    uint64_t totalWinSize_{0};
    uint64_t tailYNums_{0};
    uint64_t tailChunkNums_{0};
    uint32_t totalBlockNums_{0};
    uint32_t chunkBlockNums_{0};
    uint32_t rankDim_{0};
    uint32_t m_{0};
    uint32_t n_{0};
    uint32_t k_{0};
    uint32_t splitMCnt_{0};
    uint32_t splitM_{0};
    uint32_t chunkM_{0};
    uint32_t currentSplitId_{0};  // 当前处理的splitId，用于状态同步

    // ---- 成员变量 ----
    // 这组变量在Init()中从Tiling和MC2上下文中提取
    TPipe* tPipe_;
    GM_ADDR xAddr_;
    GM_ADDR weightAddr_;
    GM_ADDR yAddr_;
};

template<typename DataType>
__aicore__ inline void MatmulReduceScatterKernel<DataType>::Init(
    GM_ADDR x, GM_ADDR weight, GM_ADDR y,
    GM_ADDR mc2Context, GM_ADDR tilingGM, TPipe *tPipe)
{
    tPipe_ = tPipe;
    mteComm_.InitHcclContextByAddr(mc2Context);
    tilingData_ = (__gm__ MatmulReduceScatterTilingData*)tilingGM;

    auto&& tInfo = tilingData_->tilingInfo;
    m_ = tInfo.m;
    n_ = tInfo.n;
    k_ = tInfo.k;
    splitMCnt_ = tInfo.splitMCnt;
    splitM_ = tInfo.splitM;
    chunkM_ = tInfo.chunkM;

    yNums_ = tInfo.perRankLen;
    ySize_ = yNums_ * sizeof(DataType);
    
    rankDim_ = mteComm_.hcclContext_->rankDim;
    chunkNums_ = tInfo.chunkSize;
    chunkSize_ = chunkNums_ * sizeof(DataType);
    
    totalWinSize_ = mteComm_.hcclContext_->winSize;

    mteComm_.m_ = m_;
    mteComm_.n_ = n_;
    mteComm_.splitM_ = splitM_;
    mteComm_.ComputeBlockParams();

    uint64_t aivNum = tInfo.aicNum * 2;

    tailChunkNums_ = BlockAlignMod(chunkNums_, mteComm_.xNumPerBlock_);
    chunkBlockNums_ = CeilDiv(chunkSize_, mteComm_.xNumPerBlock_ * sizeof(DataType));

    mteComm_.round_ = chunkBlockNums_ / aivNum;
    mteComm_.tailBlockNums_ = chunkBlockNums_ % aivNum;
    mteComm_.ComputeTailAivId(aivNum);
    
    mteComm_.tailXNums_ = tailChunkNums_;
    mteComm_.xNums_ = yNums_;
    mteComm_.chunkSize_ = chunkNums_;
    mteComm_.totalLen_ = yNums_;

    tPipe->Reset();
    mteComm_.SetBlockSize(mteComm_.xNumPerBlock_, aivNum, tailChunkNums_);
    mteComm_.InitParams();
    mteComm_.InitBuffer(tPipe);
    mteComm_.InitGMTensor(y, ySize_, chunkSize_, totalWinSize_);

    xAddr_ = x;
    weightAddr_ = weight;
    yAddr_ = y;
}

template<typename DataType>
// ExecuteMatmul: 在AIC核上构造Matmul参数并启动GroupedMatmulKernel
// Matmul输出直接写入Win Area的Data区(不经过GM中转)
__aicore__ inline void MatmulReduceScatterKernel<DataType>::ExecuteMatmul(uint32_t splitId)
{
    if ASCEND_IS_AIV {
        return;
    }
    
    uint32_t blockIdx = AscendC::GetBlockIdx();
    auto&& tInfo = tilingData_->tilingInfo;
    
    if (blockIdx >= tInfo.aicNum) {
        return;
    }
    
    GroupedMatmulFp16TilingData gmmTiling;
    gmmTiling.groupNum = rankDim_;
    gmmTiling.maxM = splitM_;
    gmmTiling.n = n_;
    gmmTiling.k = k_;
    gmmTiling.baseM = tInfo.baseM;
    gmmTiling.baseN = tInfo.baseN;
    gmmTiling.baseK = tInfo.baseK;
    uint32_t kL1Limit = (k_ > tInfo.baseK) ? tInfo.baseK : k_;
    gmmTiling.kAL1 = kL1Limit;
    gmmTiling.kBL1 = kL1Limit;
    gmmTiling.usedCoreNum = tInfo.aicNum;
    gmmTiling.dbL0C = 1;

    using AType = bfloat16_t;
    using BType = bfloat16_t;
    using CType = bfloat16_t;
    using LayoutA = layout::RowMajor;
    using LayoutB = layout::RowMajor;
    using LayoutC = layout::RowMajor;

    using ProblemShape = MatmulShape;
    using DispatchPolicy = QuantMatmulMxMultiBlockMmad;
    using BlockMmad = Block::BlockMmadFp16<DispatchPolicy, AType, LayoutA, BType, LayoutB, CType, LayoutC>;
    using BlockScheduler = Block::BlockSchedulerFp16SplitM<ProblemShape, false, false>;
    using GroupedKernel = Kernel::GroupedMatmulFp16KernelSplitM<ProblemShape, BlockMmad, BlockScheduler>;
    using Params = typename GroupedKernel::Params;

    GM_ADDR winDataAddr = mteComm_.GetWinDataAddrGm(
        mteComm_.hcclContext_->rankId,
        mteComm_.winBufferFlags_
    );

    Params params = {
        {static_cast<int64_t>(splitM_), static_cast<int64_t>(n_), static_cast<int64_t>(k_), 1},
        {xAddr_, weightAddr_, nullptr, winDataAddr},
        &gmmTiling,
        splitId,
        chunkM_,
    };

    GroupedKernel kernel;
    kernel(params);
}

template<typename DataType>
__aicore__ inline void MatmulReduceScatterKernel<DataType>::WriteMatmulStatus()
{
    if ASCEND_IS_AIC {
        return;
    }
    mteComm_.WriteStatusToWin(currentSplitId_);
}

template<typename DataType>
__aicore__ inline void MatmulReduceScatterKernel<DataType>::ReadMatmulStatus()
{
    if ASCEND_IS_AIC {
        return;
    }
    mteComm_.ReadStatus(currentSplitId_);
}

template<typename DataType>
__aicore__ inline void MatmulReduceScatterKernel<DataType>::ExecuteReduceScatter(uint32_t splitId)
{
    if ASCEND_IS_AIC {
        return;
    }

    mteComm_.ReduceGatherFromWin(splitId);
}

template<typename DataType>
// ============================================================
// Process: Split级流水编排
//   for each Split:
//     1. ExecuteMatmul(splitId)     -- AIC核: Matmul计算 -> Win Area
//     2. SyncAll(false)             -- 核间同步屏障
//     3. WriteMatmulStatus()        -- AIV核: 写完成标记
//     4. ReadMatmulStatus()         -- AIV核: 等待所有Rank完成
//     5. ExecuteReduceScatter(splitId) -- AIV核: Win Area读取+求和
// ============================================================
__aicore__ inline void MatmulReduceScatterKernel<DataType>::Process()
{
    for (uint32_t splitId = 0; splitId < splitMCnt_; ++splitId) {
        currentSplitId_ = splitId;  // 更新当前splitId供状态同步使用
        
        ExecuteMatmul(splitId);

        AscendC::SyncAll<false>();

        if ASCEND_IS_AIC {
            continue;
        }

        WriteMatmulStatus();
        ReadMatmulStatus();
        ExecuteReduceScatter(splitId);
    }
}

}

#endif

**Process 核心流程（伪代码）：**

```cpp
void Process() {
    for (int splitId = 0; splitId < splitMCnt; splitId++) {
        // === 阶段1：Matmul 计算（AIC核） ===
        ExecuteMatmul(splitId);
        // AIC核完成 splitM 行的矩阵乘，结果写入 Win Area

        // === 阶段2：全局状态同步（AIV核） ===
        WriteMatmulStatus();   // AIV核写完成标记到所有Rank
        ReadMatmulStatus();    // AIV核轮询等待所有Rank完成

        // === 阶段3：ReduceScatter（AIV核） ===
        ExecuteReduceScatter(splitId);
        // AIV核从各Rank Win Area读取 → 规约求和 → 写出
    }
}
```

**Split 循环的设计意义：**
1. **减少 Win Area 占用**：每轮输出只占 `splitM × N` 而非 `chunkM × N`
2. **流水分级并行**：Matmul(Split i) 和 ReduceScatter(Split i-1) 粗粒度交替
3. **降低同步粒度**：减小 Barrier 等待时间

**AIC/AIV 同步时间线（单个 Split 内）**：
```
AIC核（Matmul计算）                AIV核（通信协调）
────────────────────────────────────────────────────
Compute Matmul → 结果入WinArea
                                    WriteStatusToWin
                                    ReadStatus（轮询）
                                    ReduceGatherFromWin
                                    → 累加 → 输出到GM
────────────────────────────────────────────────────
```

---
## **8. Kernel 入口**

Kernel 入口文件定义 Kernel 函数、任务类型声明和 Host 侧调用接口。

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_kernel/mte_matmul_reduce_scatter_kernel.cpp
#include "kernel_operator.h"
#include "kernel/matmul_reduce_scatter_kernel.h"

using namespace AscendC;
using namespace MatmulReduceScatterImpl;

extern "C" {
    int HcomGetCommHandleByGroup(const char* groupName, void** commHandle);
    int HcclAllocComResourceByTiling(void* comm, void* stream, void* mc2Tiling, void** commContext);
}

__attribute__((always_inline)) __aicore__ __inline__ void matmul_reduce_scatter_op(
    GM_ADDR x, GM_ADDR weight, GM_ADDR y,
    GM_ADDR workspaceGM, GM_ADDR mc2Context, GM_ADDR tilingGM)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIC_1_2);
    
    TPipe pipe;
    MatmulReduceScatterKernel<bfloat16_t> op;
    op.Init(x, weight, y, mc2Context, tilingGM, &pipe);
    op.Process();
}

extern "C" __global__ __aicore__ void MatmulReduceScatter_Generic(
    GM_ADDR x, GM_ADDR weight, GM_ADDR y,
    GM_ADDR workspaceGM, GM_ADDR mc2Context, GM_ADDR tilingGM)
{
    matmul_reduce_scatter_op(x, weight, y, workspaceGM, mc2Context, tilingGM);
}

extern "C" void matmul_reduce_scatter_demo(uint32_t blockDim, void* stream,
    uint8_t* x, uint8_t* weight, uint8_t* y,
    uint8_t* workspaceGM, uint8_t* mc2Context, uint8_t* tilingGM)
{
    MatmulReduceScatter_Generic<<<blockDim, nullptr, stream>>>(
        x, weight, y, workspaceGM, mc2Context, tilingGM);
}

**入口要点：**
- **`KERNEL_TYPE_MIX_AIC_1_2`**：声明 Kernel 为 AIC/AIV 混合模式，核按 1:2 比例分区
- **`matmul_reduce_scatter_op`**：内联 Device 函数，创建 `TPipe` + `MatmulReduceScatterKernel` 对象
- **`MatmulReduceScatter_Generic`**：`__global__ __aicore__` Kernel，通过 `<<<>>>` 语法启动
- **`matmul_reduce_scatter_demo`**：C 封装函数，Host 通过此接口调用

---
## **9. Host 侧代码实现**

Host 侧负责多设备管理、HCCL 通信初始化、Tiling 构造、MC2 上下文创建和 Kernel 启动。

### **9.1 Host 工具函数**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_host/host_utils.h
#ifndef HOST_UTILS_H
#define HOST_UTILS_H

#include <cstddef>
#include <cstdint>
#include <cstdio>
#include <fcntl.h>
#include <fstream>
#include <iostream>
#include <limits.h>
#include <string>
#include <sys/stat.h>
#include <unistd.h>

#include "acl/acl.h"

#define ERROR_LOG(fmt, args...) fprintf(stdout, "[ERROR]  " fmt "\n", ##args)

template <typename T>
inline T CeilDiv(T a, T b)
{
    if (b == 0) {
        return a;
    }
    return (a + b - 1) / b;
}

template <typename T>
inline T Align(T a, T b)
{
    return CeilDiv(a, b) * b;
}

inline bool ReadFile(const std::string& filePath, size_t& fileSize, void* buffer, size_t bufferSize)
{
    if (buffer == nullptr) {
        ERROR_LOG("Read file failed. buffer is nullptr");
        return false;
    }
    struct stat sBuf;
    int fileStatus = stat(filePath.data(), &sBuf);
    if (fileStatus == -1) {
        ERROR_LOG("failed to get file");
        return false;
    }
    if (S_ISREG(sBuf.st_mode) == 0) {
        ERROR_LOG("%s is not a file, please enter a file", filePath.c_str());
        return false;
    }

    std::ifstream file;
    file.open(filePath, std::ios::binary);
    if (!file.is_open()) {
        ERROR_LOG("Open file failed. path = %s", filePath.c_str());
        return false;
    }

    std::filebuf* buf = file.rdbuf();
    size_t size = buf->pubseekoff(0, std::ios::end, std::ios::in);
    if (size == 0) {
        ERROR_LOG("file size is 0");
        file.close();
        return false;
    }
    if (size > bufferSize) {
        ERROR_LOG("file size is larger than buffer size");
        file.close();
        return false;
    }
    buf->pubseekpos(0, std::ios::in);
    buf->sgetn(static_cast<char*>(buffer), size);
    fileSize = size;
    file.close();
    return true;
}

inline bool WriteFile(const std::string& filePath, const void* buffer, size_t size)
{
    if (buffer == nullptr) {
        ERROR_LOG("Write file failed. buffer is nullptr");
        return false;
    }

    int fd = open(filePath.c_str(), O_RDWR | O_CREAT | O_TRUNC, S_IRUSR | S_IWRITE);
    if (fd < 0) {
        ERROR_LOG("Open file failed. path = %s", filePath.c_str());
        return false;
    }

    size_t writeSize = write(fd, buffer, size);
    (void)close(fd);
    if (writeSize != size) {
        ERROR_LOG("Write file Failed.");
        return false;
    }

    return true;
}

#endif // HOST_UTILS_H

### **9.2 Host 主程序**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_host/mte_matmul_reduce_scatter_host.cpp
#include <thread>
#include <iostream>
#include <vector>
#include <string>
#include <cstring>
#include <cerrno>
#include <getopt.h>
#include <fstream>
#include <unistd.h>
#include <sys/file.h>
#include <sys/stat.h>
#include <cmath>
#include <cstdlib>
#include <numeric>
#include <algorithm>
#include <memory>
#include "acl/acl.h"
#include "hccl/hccl_types.h"
#include "hccl/hccl_comm.h"
#include "hccl/hcom.h"
#include "securec.h"
#include "adv_api/hccl/hccl_tiling.h"
#include "adv_api/kernel_tiling.h"
#include "platform/platform_ascendc.h"
#include "host_utils.h"

using namespace std;

static constexpr size_t BF16_SIZE = 2;
static constexpr int64_t DEV_NUM = 4;

#define CHECK_RET(cond, return_expr) \
    do {                             \
        if (!(cond)) {               \
            return_expr;             \
        }                            \
    } while (0)

#define LOG_PRINT(message, ...)         \
    do {                                \
        printf(message, ##__VA_ARGS__); \
    } while (0)

int streamWithTimeout = 10000;
int64_t gAivNum = 0;
int gMinSplitM = 128;  // 默认等于 baseM，必须是 128 的倍数
int gM = 1024;
int gN = 1024;
int gK = 1024;
bool gPerfMode = false;
constexpr int PERF_LOOP_COUNT = 10;
uint32_t gSplitMCnt = 0;   // 自动计算后的 splitMCnt
uint32_t gSplitM = 0;      // 自动计算后的 splitM
int64_t gWorkSpaceSize = 256 * 1024 * 1024;
int64_t gHcclBufferSize = []() {
    const char* envStr = getenv("HCCL_BUFFSIZE");
    return (envStr != nullptr) ? atoll(envStr) : 200;
}();
std::string gOutputDir = ".";

extern "C" void matmul_reduce_scatter_demo(uint32_t blockDim, void* stream,
    uint8_t* x, uint8_t* weight, uint8_t* y,
    uint8_t* workspaceGM, uint8_t* mc2Context, uint8_t* tilingGM);

extern "C" HcclResult HcclAllocComResourceByTiling(HcclComm comm, void *stream, void *mc2Tiling, void **commContext);

extern "C" uint32_t GetCoreNumForMixVectorCore(uint32_t *aiCoreNum, uint32_t *vectorCoreNum);

struct MatmulReduceScatterTilingInfoHost {
    uint64_t perRankLen;
    uint64_t chunkSize;
    uint32_t m;
    uint32_t n;
    uint32_t k;
    uint32_t aicNum;
    uint32_t baseM;
    uint32_t baseN;
    uint32_t baseK;
    uint32_t splitMCnt;
    uint32_t splitM;
    uint32_t chunkM;
};

struct MatmulReduceScatterTilingDataHost {
    Mc2InitTiling mc2InitTiling;
    Mc2CcTiling mc2CcTiling;
    MatmulReduceScatterTilingInfoHost tilingInfo;
};

struct HcclA5OpResParamHost {
    uint64_t workSpace;
    uint64_t workSpaceSize;
    uint32_t rankId;
    uint32_t rankDim;
    uint64_t winSize;
    uint64_t windowsIn[64];
    uint64_t windowsOut[64];
    uint64_t xnAddr;
    uint64_t ckeAddr;
    uint64_t msAddr;
    uint64_t msSize;
};

int CreateTilingDataAndContext(const char* hcomName, aclrtStream stream,
    void **deviceTilingAddr, void **deviceContextAddr)
{
    MatmulReduceScatterTilingDataHost *tilingData = new MatmulReduceScatterTilingDataHost();
    if (tilingData == nullptr) {
        LOG_PRINT("[ERROR] tilingData is nullptr\n");
        return -1;
    }
    memset(tilingData, 0, sizeof(MatmulReduceScatterTilingDataHost));

auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    uint32_t aicNum = 28;

    // 校验1: M 必须能整除 rankSize
    if (gM % DEV_NUM != 0) {
        LOG_PRINT("[ERROR] M=%d must be divisible by rankSize=%d\n", gM, DEV_NUM);
        delete tilingData;
        return -1;
    }

    // 校验2: minSplitM 必须是 128 的倍数
    if (gMinSplitM % 128 != 0 || gMinSplitM < 128) {
        LOG_PRINT("[ERROR] minSplitM=%d must be multiple of 128 and >= 128\n", gMinSplitM);
        delete tilingData;
        return -1;
    }

    uint64_t perRankLen = static_cast<uint64_t>(gM) * static_cast<uint64_t>(gN);
    uint64_t chunkSize = perRankLen / DEV_NUM;
    uint32_t chunkM = gM / DEV_NUM;

    // 校验3: chunkM 必须能整除 minSplitM
    if (chunkM % gMinSplitM != 0) {
        LOG_PRINT("[ERROR] chunkM=%u cannot divide minSplitM=%d evenly\n", chunkM, gMinSplitM);
        delete tilingData;
        return -1;
    }

    // 自动计算 splitMCnt
    uint32_t splitMCnt = chunkM / gMinSplitM;
    if (splitMCnt < 1) {
        LOG_PRINT("[ERROR] chunkM=%u < minSplitM=%d, cannot split\n", chunkM, gMinSplitM);
        delete tilingData;
        return -1;
    }

    uint32_t splitM = chunkM / splitMCnt;

    // 保存到全局变量供 LaunchOneThread 使用
    gSplitMCnt = splitMCnt;
    gSplitM = splitM;

    LOG_PRINT("[INFO] Auto calculated: splitMCnt=%u, splitM=%u, chunkM=%u\n", splitMCnt, splitM, chunkM);

    tilingData->tilingInfo.perRankLen = perRankLen;
    tilingData->tilingInfo.chunkSize = chunkSize;
    tilingData->tilingInfo.m = gM;
    tilingData->tilingInfo.n = gN;
    tilingData->tilingInfo.k = gK;
    tilingData->tilingInfo.aicNum = aicNum;
    tilingData->tilingInfo.baseM = std::min(static_cast<uint32_t>(splitM), 128U);
    tilingData->tilingInfo.baseN = std::min(static_cast<uint32_t>(gN), 128U);
    tilingData->tilingInfo.baseK = std::min(static_cast<uint32_t>(gK), 128U);
    tilingData->tilingInfo.splitMCnt = splitMCnt;
    tilingData->tilingInfo.splitM = splitM;
    tilingData->tilingInfo.chunkM = chunkM;

    LOG_PRINT("[INFO] Platform: aicNum=%u, aivNum=%ld\n", aicNum, gAivNum);
    LOG_PRINT("[INFO] ReduceScatter: perRankLen=%lu, chunkSize=%lu\n", perRankLen, chunkSize);

    AscendC::Mc2CcTilingConfig mc2CcTilingConfig(hcomName, 6, "AlltoAll=level0:fullmesh;level1:pairwise");
    mc2CcTilingConfig.SetCommEngine(3);

    int ret = mc2CcTilingConfig.GetTiling(tilingData->mc2InitTiling);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] GetTiling mc2InitTiling failed. ret = %d\n", ret); delete tilingData; return ret);
    ret = mc2CcTilingConfig.GetTiling(tilingData->mc2CcTiling);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] GetTiling mc2CcTiling failed. ret = %d\n", ret); delete tilingData; return ret);

    int tilingSize = sizeof(MatmulReduceScatterTilingDataHost);
    ret = aclrtMalloc(deviceTilingAddr, tilingSize, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMalloc TilingData failed. ret: %d\n", ret); delete tilingData; return ret);
    ret = aclrtMemcpy(*deviceTilingAddr, tilingSize, tilingData, tilingSize, ACL_MEMCPY_HOST_TO_DEVICE);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMemcpy TilingData failed. ret: %d\n", ret); delete tilingData; return ret);

    HcclComm commHandle;
    ret = HcomGetCommHandleByGroup(hcomName, &commHandle);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] HcomGetCommHandleByGroup failed. ret = %d\n", ret); delete tilingData; return ret);

    void *mc2Context = nullptr;
    ret = HcclAllocComResourceByTiling(commHandle, stream, tilingData, &mc2Context);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] HcclAllocComResourceByTiling failed. ret = %d\n", ret); delete tilingData; return ret);

    if (mc2Context == nullptr) {
        LOG_PRINT("[ERROR] mc2Context is nullptr\n");
        delete tilingData;
        return -1;
    }

    constexpr size_t mc2ContextSize = sizeof(HcclA5OpResParamHost);
    ret = aclrtMalloc(deviceContextAddr, mc2ContextSize, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMalloc Mc2Context failed. ret: %d\n", ret); delete tilingData; return ret);
    ret = aclrtMemcpy(*deviceContextAddr, mc2ContextSize, mc2Context, mc2ContextSize, ACL_MEMCPY_HOST_TO_DEVICE);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMemcpy Mc2Context failed. ret: %d\n", ret); delete tilingData; return ret);

    delete tilingData;
    return ACL_SUCCESS;
}

struct Args {
    uint32_t rankId;
    uint32_t rankDim;
    HcclComm hcclComm;
    aclrtStream stream;
    aclrtContext context;
};

int LaunchOneThread(Args &args)
{
    int ret = aclrtSetCurrentContext(args.context);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtSetCurrentContext failed. ret = %d\n", ret); return ret);

    char hcomName[128] = {0};
    ret = HcclGetCommName(args.hcclComm, hcomName);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] HcclGetCommName failed. ret = %d\n", ret); return ret);
    LOG_PRINT("[INFO] rank = %d, hcomName = %s\n", args.rankId, hcomName);

    uint32_t aicNum = 28;

    void *xDeviceAddr = nullptr;
    void *weightDeviceAddr = nullptr;
    void *yDeviceAddr = nullptr;
    void *workspaceAddr = nullptr;
    void *tilingAddr = nullptr;
    void *mc2ContextAddr = nullptr;

    ret = CreateTilingDataAndContext(hcomName, args.stream, &tilingAddr, &mc2ContextAddr);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] CreateTilingDataAndContext failed. ret = %d\n", ret); return ret);

    uint64_t perRankOutLen = static_cast<uint64_t>(gM) * static_cast<uint64_t>(gN);
    uint64_t chunkOutLen = perRankOutLen / args.rankDim;
    uint64_t chunkOutBytes = chunkOutLen * BF16_SIZE;

    uint64_t xBytes = static_cast<uint64_t>(gM) * static_cast<uint64_t>(gK) * BF16_SIZE;
    uint64_t weightBytes = static_cast<uint64_t>(gN) * static_cast<uint64_t>(gK) * BF16_SIZE;
    // ReduceScatter输出大小 = chunkOutBytes（而非完整Matmul输出）
    uint64_t yBytes = chunkOutBytes;

    LOG_PRINT("[INFO] rank=%d ReduceScatter output: chunkOutBytes=%lu\n", args.rankId, chunkOutBytes);

    ret = aclrtMalloc(&xDeviceAddr, xBytes, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMalloc x failed. ret: %d\n", ret); return ret);

    ret = aclrtMalloc(&weightDeviceAddr, weightBytes, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMalloc weight failed. ret: %d\n", ret); return ret);

    ret = aclrtMalloc(&yDeviceAddr, yBytes, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMalloc y failed. ret: %d\n", ret); return ret);

    if (gWorkSpaceSize > 0) {
        ret = aclrtMalloc(&workspaceAddr, gWorkSpaceSize, ACL_MEM_MALLOC_HUGE_FIRST);
        CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMalloc workspace failed. ret: %d\n", ret); return ret);
    }

    std::string inputDir = gOutputDir + "/input";
    
    std::string testPath = inputDir + "/input_a_rank" + std::to_string(args.rankId) + ".bin";
    std::ifstream testA(testPath, std::ios::binary);
    if (!testA.is_open()) {
        LOG_PRINT("[ERROR] Cannot open file: %s\n", testPath.c_str());
        return -1;
    }
    testA.close();

    std::vector<uint8_t> hostX(xBytes, 0);
    std::vector<uint8_t> hostWeight(weightBytes, 0);

    size_t xFileSize = xBytes;
    size_t weightFileSize = weightBytes;

    CHECK_RET(ReadFile(inputDir + "/input_a_rank" + std::to_string(args.rankId) + ".bin", xFileSize, hostX.data(), xBytes),
              LOG_PRINT("[ERROR] Failed to read input_a_rank%d.bin.\n", args.rankId); return -1);
    CHECK_RET(ReadFile(inputDir + "/input_b.bin", weightFileSize, hostWeight.data(), weightBytes),
              LOG_PRINT("[ERROR] Failed to read input_b.bin.\n"); return -1);

    ret = aclrtMemcpyAsync(xDeviceAddr, xBytes, hostX.data(), xBytes, ACL_MEMCPY_HOST_TO_DEVICE, args.stream);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMemcpy x failed. ret: %d\n", ret); return ret);

    ret = aclrtMemcpyAsync(weightDeviceAddr, weightBytes, hostWeight.data(), weightBytes, ACL_MEMCPY_HOST_TO_DEVICE, args.stream);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMemcpy weight failed. ret: %d\n", ret); return ret);

    LOG_PRINT("[INFO] rank=%d Launching kernel: splitMCnt=%u, splitM=%u, chunkM=%d, M=%d, N=%d, K=%d, rankDim=%d, aicNum=%u\n",
        args.rankId, gSplitMCnt, gSplitM, static_cast<int>(gM / DEV_NUM), gM, gN, gK, args.rankDim, aicNum);

    int loopCount = gPerfMode ? PERF_LOOP_COUNT : 1;
    for (int i = 0; i < loopCount; i++) {
        matmul_reduce_scatter_demo(aicNum, args.stream,
            (uint8_t*)xDeviceAddr, (uint8_t*)weightDeviceAddr, (uint8_t*)yDeviceAddr,
            (uint8_t*)workspaceAddr, (uint8_t*)mc2ContextAddr, (uint8_t*)tilingAddr);

        ret = aclrtSynchronizeStreamWithTimeout(args.stream, streamWithTimeout);
        CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtSynchronizeStreamWithTimeout failed. ret = %d\n", ret); return ret);
    }
    LOG_PRINT("[INFO] rank=%d kernel executed successfully\n", args.rankId);

    std::vector<uint8_t> hostY(chunkOutBytes, 0);
    ret = aclrtMemcpy(hostY.data(), chunkOutBytes, yDeviceAddr, chunkOutBytes, ACL_MEMCPY_DEVICE_TO_HOST);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtMemcpy y failed. ret: %d\n", ret); return ret);

    std::string outFile = gOutputDir + "/output/npu_out_rank" + std::to_string(args.rankId) + ".bin";
    WriteFile(outFile, hostY.data(), chunkOutBytes);
    LOG_PRINT("[INFO] rank=%d output written to %s\n", args.rankId, outFile.c_str());

    aclrtFree(xDeviceAddr);
    aclrtFree(weightDeviceAddr);
    aclrtFree(yDeviceAddr);
    if (workspaceAddr) aclrtFree(workspaceAddr);
    aclrtFree(mc2ContextAddr);
    aclrtFree(tilingAddr);

    HcclCommDestroy(args.hcclComm);
    aclrtDestroyStream(args.stream);
    aclrtResetDevice(args.rankId);
    aclrtDestroyContext(args.context);

    return ACL_SUCCESS;
}

void PrintUsageHelp(const char* progName)
{
    std::cout << "Usage: " << progName << " [options]\n"
              << "Options:\n"
              << "  --m <value>         M dimension (default: 1024)\n"
              << "  --n <value>         N dimension (default: 1024)\n"
              << "  --k <value>         K dimension (default: 1024)\n"
              << "  --min_split_m <value>  Minimum splitM, must be multiple of 128 (default: 128)\n"
              << "  --output_dir <dir>  Output directory (default: .)\n"
              << "  --help              Show this help\n";
}

int main(int argc, char *argv[])
{
    static const struct option longOptions[] = {
        {"m",           1, 0, 'm'},
        {"n",           1, 0, 'n'},
        {"k",           1, 0, 'k'},
        {"min_split_m", 1, 0, 's'},
        {"output_dir",  1, 0, 'o'},
        {"perf_mode",   0, 0, 'p'},
        {"help",        0, 0, 'h'},
        {0, 0, 0, 0}
    };

    int opt;
    while ((opt = getopt_long(argc, argv, "m:n:k:s:o:ph", longOptions, nullptr)) != -1) {
        switch (opt) {
            case 'm':
                if (optarg) {
                    gM = atoi(optarg);
                    if (gM <= 0) {
                        LOG_PRINT("[ERROR] m must be positive integer. Got: %d\n", gM);
                        return -1;
                    }
                }
                break;
            case 'n':
                if (optarg) {
                    gN = atoi(optarg);
                    if (gN <= 0) {
                        LOG_PRINT("[ERROR] n must be positive integer. Got: %d\n", gN);
                        return -1;
                    }
                }
                break;
            case 'k':
                if (optarg) {
                    gK = atoi(optarg);
                    if (gK <= 0) {
                        LOG_PRINT("[ERROR] k must be positive integer. Got: %d\n", gK);
                        return -1;
                    }
                }
                break;
            case 's':
                if (optarg) {
                    gMinSplitM = atoi(optarg);
                    if (gMinSplitM % 128 != 0 || gMinSplitM < 128) {
                        LOG_PRINT("[ERROR] min_split_m must be multiple of 128 and >= 128. Got: %d\n", gMinSplitM);
                        return -1;
                    }
                }
                break;
            case 'o': if (optarg) gOutputDir = optarg; break;
            case 'p': gPerfMode = true; break;
            case 'h': PrintUsageHelp(argv[0]); return 0;
        }
    }

    LOG_PRINT("[INFO] Config: minSplitM=%d, M=%d, N=%d, K=%d, DEV_NUM=%ld\n",
              gMinSplitM, gM, gN, gK, DEV_NUM);

    int ret = aclInit(nullptr);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclInit failed. ret = %d\n", ret); return ret);

    aclrtStream stream[DEV_NUM];
    aclrtContext context[DEV_NUM];
    for (uint32_t rankId = 0; rankId < DEV_NUM; rankId++) {
        ret = aclrtSetDevice(rankId);
        CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtSetDevice failed. ret = %d\n", ret); return ret);
        ret = aclrtCreateContext(&context[rankId], rankId);
        CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtCreateContext failed. ret = %d\n", ret); return ret);
        ret = aclrtCreateStream(&stream[rankId]);
        CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] aclrtCreateStream failed. ret = %d\n", ret); return ret);
    }

    uint32_t aicCoreNum = 0;
    uint32_t aivCoreNum = 0;
    ret = GetCoreNumForMixVectorCore(&aicCoreNum, &aivCoreNum);
    if (ret == 0 && aivCoreNum > 0) {
        gAivNum = aivCoreNum;
        LOG_PRINT("[INFO] GetCoreNumForMixVectorCore: aicCoreNum=%u, aivCoreNum=%u\n", aicCoreNum, aivCoreNum);
    } else {
        gAivNum = 56;
        LOG_PRINT("[WARN] GetCoreNumForMixVectorCore failed, fallback gAivNum=%ld\n", gAivNum);
    }

    int32_t devices[DEV_NUM];
    for (int i = 0; i < DEV_NUM; i++) {
        devices[i] = i;
    }

    HcclComm comms[DEV_NUM];
    ret = HcclCommInitAll(DEV_NUM, devices, comms);
    CHECK_RET(ret == ACL_SUCCESS, LOG_PRINT("[ERROR] HcclCommInitAll failed. ret = %d\n", ret); return ret);

    Args args[DEV_NUM];
    std::vector<std::unique_ptr<std::thread>> threads(DEV_NUM);
    int returnCodes[DEV_NUM] = {0};

    for (uint32_t rankId = 0; rankId < DEV_NUM; rankId++) {
        args[rankId].rankId = rankId;
        args[rankId].rankDim = DEV_NUM;
        args[rankId].hcclComm = comms[rankId];
        args[rankId].context = context[rankId];
        args[rankId].stream = stream[rankId];
        threads[rankId].reset(new(std::nothrow) std::thread([&args, &returnCodes, rankId]() {
            returnCodes[rankId] = LaunchOneThread(args[rankId]);
        }));
    }

    for (uint32_t rankId = 0; rankId < DEV_NUM; rankId++) {
        threads[rankId]->join();
    }

    aclFinalize();

    int finalRet = 0;
    for (uint32_t rankId = 0; rankId < DEV_NUM; rankId++) {
        if (returnCodes[rankId] != 0) {
            finalRet = 1;
        }
    }

    if (finalRet == 0) {
        printf("========== All ranks PASSED ==========\n");
    } else {
        printf("========== Some ranks FAILED ==========\n");
    }

    return finalRet;
}

**Host 侧关键流程：**

**1. 参数配置（main 函数）：**
- 命令行解析：支持 `--m/--n/--k/--min_split_m/--output_dir/--perf_mode`
- 默认参数：M=N=K=1024，DEV_NUM=4

**2. 多设备初始化：**
- `aclInit` → 为每个 Device 创建 Context 和 Stream
- `GetCoreNumForMixVectorCore` 查询可用 AIC/AIV 核数
- `HcclCommInitAll` 初始化 4 Rank 的通信域

**3. CreateTilingDataAndContext（Tiling 构造核心）：**
- 三重校验：M % rankSize == 0、minSplitM 是 128 的倍数、chunkM % minSplitM == 0
- 自动推导：`chunkM = M/DEV_NUM`、`splitMCnt = chunkM/minSplitM`、`splitM = chunkM/splitMCnt`
- MC2 Tiling：`Mc2CcTilingConfig` 配置通信拓扑 "AlltoAll=level0:fullmesh;level1:pairwise"
- 上下文创建：`HcclAllocComResourceByTiling` 分配 Win Area

**4. LaunchOneThread（单 Rank 执行）：**
- 创建 Tiling 和通信上下文
- 分配 Device 内存（x[M*K]、weight[N*K]、y[chunkOutBytes]、workspace[256MB]）
- 异步加载输入数据 → 启动 Kernel → 回读 ReduceScatter 输出
- 注意：**输出大小 yBytes = chunkOutBytes**（不是完整的 Matmul 输出！）

**5. 多线程并发：** 每个 Rank 一个 `std::thread`，并发下发任务到不同 Device。

---
## **10. 构建与运行**

### **10.1 CMakeLists.txt**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(MatmulReduceScatter)

set(CMAKE_EXPORT_COMPILE_COMMANDS ON)

if(NOT CMAKE_BUILD_TYPE)
    set(CMAKE_BUILD_TYPE "Release" CACHE STRING "Build type Release/Debug" FORCE)
endif()

set(SOC_VERSION "ascend950pr_957c" CACHE STRING "system on chip type")

if(DEFINED ENV{ASCEND_HOME_PATH})
    set(ASCEND_CANN_PACKAGE_PATH $ENV{ASCEND_HOME_PATH})
else()
    message(FATAL_ERROR "ASCEND_HOME_PATH not set, please configure CANN environment first")
endif()

set(ASCENDC_CMAKE_DIR "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake")
if(NOT EXISTS "${ASCENDC_CMAKE_DIR}")
    set(ASCENDC_CMAKE_DIR "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake")
endif()
if(NOT EXISTS "${ASCENDC_CMAKE_DIR}")
    set(ASCENDC_CMAKE_DIR "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake")
endif()
if(NOT EXISTS "${ASCENDC_CMAKE_DIR}")
    message(FATAL_ERROR "ascendc_kernel_cmake not found, check CANN at: ${ASCEND_CANN_PACKAGE_PATH}")
endif()
include("${ASCENDC_CMAKE_DIR}/ascendc.cmake")

set(MTE_MATMUL_REDUCE_SCATTER_DIR ${CMAKE_CURRENT_SOURCE_DIR})
get_filename_component(TENSOR_API_DIR "${CMAKE_CURRENT_SOURCE_DIR}/../../../../teams/ops-direct-invoke" ABSOLUTE)

set(INCLUDE_DIRS
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel/include
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel/include/kernel
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel/include/block
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel/include/tile
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel/include/tiling
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel/include/utils
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_host
    ${TENSOR_API_DIR}/asc-devkit
    ${TENSOR_API_DIR}/asc-devkit/include
    ${TENSOR_API_DIR}/asc-devkit/include/basic_api
)

set(KERNEL_SRC
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_kernel/mte_matmul_reduce_scatter_kernel.cpp
)

ascendc_library(matmul_reduce_scatter_kernel SHARED ${KERNEL_SRC})
ascendc_include_directories(matmul_reduce_scatter_kernel PRIVATE ${INCLUDE_DIRS})

add_executable(matmul_reduce_scatter
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_host/mte_matmul_reduce_scatter_host.cpp
)
target_compile_options(matmul_reduce_scatter PRIVATE -O2 -std=c++17 -D_GLIBCXX_USE_CXX11_ABI=0)
target_compile_definitions(matmul_reduce_scatter PRIVATE SOC_VERSION="${SOC_VERSION}")
target_include_directories(matmul_reduce_scatter PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/pkg_inc
    ${ASCEND_CANN_PACKAGE_PATH}/include
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/pkg_inc
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/asc/include
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/asc/include/adv_api
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/asc/include/tiling
    ${INCLUDE_DIRS}
)
target_link_libraries(matmul_reduce_scatter PRIVATE
    matmul_reduce_scatter_kernel
    tiling_api
    platform
    pthread
    hccl
    hcomm
    ascendc_runtime
)

install(TARGETS matmul_reduce_scatter matmul_reduce_scatter_kernel
    RUNTIME DESTINATION .
    LIBRARY DESTINATION .
)

install(DIRECTORY
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/op_host/scripts
    DESTINATION .
)

install(FILES
    ${MTE_MATMUL_REDUCE_SCATTER_DIR}/run_multi_data.sh
    DESTINATION .
)

**CMake 配置要点：**
- `SOC_VERSION = ascend950pr_957c`
- `ascendc_library` 编译 Kernel 为 `.so` 动态库
- `add_executable` 编译 Host 为可执行文件
- 链接库：`matmul_reduce_scatter_kernel`、`tiling_api`、`hccl`、`hcomm`、`ascendc_runtime`、`pthread`

### **10.2 编译脚本**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/build.sh
#!/bin/bash

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
BUILD_DIR="${SCRIPT_DIR}/build"

if [ -z "$ASCEND_HOME_PATH" ]; then
    echo "ERROR: CANN environment not found. Please set ASCEND_HOME_PATH or source set_env.sh"
    exit 1
fi

echo "========== Building Matmul+ReduceScatter =========="
echo "Build directory: ${BUILD_DIR}"

rm -rf "${BUILD_DIR}"
mkdir -p "${BUILD_DIR}"
cd "${BUILD_DIR}"

cmake -DCMAKE_INSTALL_PREFIX="${BUILD_DIR}" ..
if [ $? -ne 0 ]; then
    echo "ERROR: cmake failed"
    exit 1
fi

make -j8 install
if [ $? -ne 0 ]; then
    echo "ERROR: make failed"
    exit 1
fi

echo "========== Build Success =========="
echo "Executables location: ${BUILD_DIR}"
echo "  - matmul_reduce_scatter"
echo "  - libmatmul_reduce_scatter_kernel.so"
echo "  - scripts/"
echo "  - run_multi_data.sh"
echo ""
echo "To run: cd ${BUILD_DIR} && bash run_multi_data.sh"

### **10.3 运行脚本**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/run_multi_data.sh
#!/bin/bash

BUILD_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"

if [ -n "${ASCEND_HOME_PATH}" ]; then
    ASCEND_DIR="${ASCEND_HOME_PATH}"
elif [ -n "${ASCEND_DIR}" ]; then
    ASCEND_DIR="${ASCEND_DIR}"
else
    echo "ERROR: ASCEND_HOME_PATH or ASCEND_DIR environment variable not set"
    echo "Please set ASCEND_HOME_PATH or ASCEND_DIR to your Ascend installation path"
    echo "Example: source /path/to/Ascend/cann/set_env.sh"
    exit 1
fi

export LD_LIBRARY_PATH="${BUILD_DIR}:${ASCEND_DIR}/lib64:${LD_LIBRARY_PATH}"

EXECUTABLE="${BUILD_DIR}/matmul_reduce_scatter"
KERNEL_LIB="${BUILD_DIR}/libmatmul_reduce_scatter_kernel.so"
SCRIPTS_DIR="${BUILD_DIR}/scripts"

RANK_SIZE=4
export HCCL_BUFFSIZE=2048

if [ ! -f "$EXECUTABLE" ]; then
    echo "ERROR: Executable not found: $EXECUTABLE"
    echo "Please build the project first: bash build.sh"
    exit 1
fi

if [ ! -f "$KERNEL_LIB" ]; then
    echo "ERROR: Kernel library not found: $KERNEL_LIB"
    exit 1
fi

run_test_case() {
    local case_name=$1
    local m=$2
    local n=$3
    local k=$4
    
    local work_dir="${BUILD_DIR}/work_${case_name}"
    
    echo ""
    echo "=========================================="
    echo "Test Case: ${case_name}"
    echo "Parameters: m=${m}, n=${n}, k=${k}"
    echo "=========================================="
    
    mkdir -p "${work_dir}/input" "${work_dir}/output"
    
    cd "${work_dir}"
    python3 "${SCRIPTS_DIR}/gen_data.py" ${RANK_SIZE} ${m} ${k} ${n}
    
    if [ $? -ne 0 ]; then
        echo "ERROR: [${case_name}] Data generation failed"
        return 1
    fi
    
    echo "Executing Matmul+ReduceScatter with pipeline optimization..."
    echo "ReduceScatter output size per rank: M*N/rankSize = $((m * n / RANK_SIZE)) elements"
    
    ${EXECUTABLE} \
        --m ${m} \
        --n ${n} \
        --k ${k} \
        --output_dir "${work_dir}"
    
    if [ $? -ne 0 ]; then
        echo "ERROR: [${case_name}] Execution failed"
        return 1
    fi
    
    echo "Verifying results..."
    python3 "${SCRIPTS_DIR}/verify_result.py" ${RANK_SIZE} ${m} ${k} ${n}
    
    if [ $? -ne 0 ]; then
        echo "ERROR: [${case_name}] Verification failed"
        return 1
    fi
    
    echo "[${case_name}] PASSED"
    return 0
}

TEST_CASES=(
    "base_1024:1024:1024:1024"
    "large_2048:2048:2048:2048"
    "large_4096:4096:4096:4096"
)

TOTAL_PASSED=0
TOTAL_FAILED=0

for case in "${TEST_CASES[@]}"; do
    IFS=':' read -r name m n k <<< "$case"
    
    if run_test_case "$name" "$m" "$n" "$k"; then
        TOTAL_PASSED=$((TOTAL_PASSED + 1))
    else
        TOTAL_FAILED=$((TOTAL_FAILED + 1))
    fi
done

echo ""
echo "=========================================="
echo "Test Summary"
echo "=========================================="
echo "Total:  ${TOTAL_PASSED} passed, ${TOTAL_FAILED} failed"

if [ ${TOTAL_FAILED} -eq 0 ]; then
    echo "All test cases PASSED!"
    exit 0
else
    echo "Some test cases FAILED!"
    exit 1
fi

运行脚本特点：
- 内置 3 组测试 Case（1024/2048/4096）
- 全自动流程：生成数据 → 执行算子 → 精度验证
- `HCCL_BUFFSIZE=2048` 设置 Win Area 缓冲区大小

---
## **11. 测试与验证**

### **11.1 测试数据生成**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_host/scripts/gen_data.py
import torch
import numpy as np
import struct
import sys
import os

def ceil_div(a, b):
    return (a + b - 1) // b

def align(a, b):
    return ceil_div(a, b) * b

def gen_golden_data(rank_size, m, k, n, all_one=False):
    os.makedirs("./input", exist_ok=True)
    os.makedirs("./output", exist_ok=True)

    if all_one:
        b_bf16 = torch.ones(k * n, dtype=torch.bfloat16)
        b_bytes = b_bf16.view(torch.uint16).numpy().tobytes()
    else:
        np.random.seed(42)
        b_data = np.random.randn(k, n).astype(np.float32) * 0.1
        b_bf16 = torch.from_numpy(b_data).to(torch.bfloat16)
        b_bytes = b_bf16.view(torch.uint16).numpy().tobytes()
    with open("./input/input_b.bin", "wb") as f:
        f.write(b_bytes)

    matmul_outputs = []
    for rank_id in range(rank_size):
        if all_one:
            a_bf16 = torch.ones(m * k, dtype=torch.bfloat16)
            a_bytes = a_bf16.view(torch.uint16).numpy().tobytes()
        else:
            np.random.seed(42 + rank_id)
            a_data = np.random.randn(m, k).astype(np.float32) * 0.1
            a_bf16 = torch.from_numpy(a_data).to(torch.bfloat16)
            a_bytes = a_bf16.view(torch.uint16).numpy().tobytes()
        with open(f"./input/input_a_rank{rank_id}.bin", "wb") as f:
            f.write(a_bytes)

        if all_one:
            cpu_output = np.full((m, n), k, dtype=np.float32)
        else:
            a_tensor = torch.from_numpy(a_data).to(torch.bfloat16)
            b_tensor = torch.from_numpy(b_data).to(torch.bfloat16)
            cpu_output = (a_tensor.float() @ b_tensor.float()).numpy()

        matmul_outputs.append(cpu_output)
        cpu_out_bf16 = torch.from_numpy(cpu_output).to(torch.bfloat16)
        with open(f"./output/cpu_matmul_output_rank{rank_id}.bin", "wb") as f:
            f.write(cpu_out_bf16.view(torch.uint16).numpy().tobytes())

    # ReduceScatter: 对所有rank的Matmul输出按chunk分区求和
    chunk_size = m * n // rank_size
    
    if all_one:
        expected_value = k * rank_size  # 全1时，每个元素 = Σ(1×k) × rankSize = k × rankSize
        print(f"ReduceScatter (all_one test): chunk_size = {chunk_size}, expected_value = {expected_value}")
    else:
        print(f"ReduceScatter: chunk_size = {chunk_size} elements (m*n/rankSize = {m}*{n}/{rank_size})")
    
    for rank_id in range(rank_size):
        chunk_start = rank_id * chunk_size
        chunk_end = (rank_id + 1) * chunk_size
        
        reduced_chunk = np.zeros(chunk_size, dtype=np.float32)
        for src_rank in range(rank_size):
            matmul_flat = matmul_outputs[src_rank].flatten()
            reduced_chunk += matmul_flat[chunk_start:chunk_end]
        
        reduced_bf16 = torch.from_numpy(reduced_chunk).to(torch.bfloat16)
        with open(f"./output/cpu_reducescatter_rank{rank_id}.bin", "wb") as f:
            f.write(reduced_bf16.view(torch.uint16).numpy().tobytes())
        
        print(f"  Rank {rank_id}: chunk [{chunk_start}:{chunk_end}], min={reduced_chunk.min():.2f}, max={reduced_chunk.max():.2f}")

    # 生成完整的Matmul输出拼接（用于可视化）
    all_matmul_output = np.concatenate(matmul_outputs, axis=0)
    all_matmul_bf16 = torch.from_numpy(all_matmul_output).to(torch.bfloat16)
    with open("./output/cpu_all_matmul_output.bin", "wb") as f:
        f.write(all_matmul_bf16.view(torch.uint16).numpy().tobytes())

    print(f"\nGenerated files in ./input/:")
    for f in os.listdir("./input"):
        print(f"  {f}")
    print(f"\nGenerated golden files in ./output/:")
    for f in os.listdir("./output"):
        print(f"  {f}")
    print(f"\nReduceScatter configuration:")
    print(f"  rank_size: {rank_size}")
    print(f"  chunk_size per rank: {chunk_size} elements = {chunk_size * 2} bytes")
    print(f"  total Matmul output: {m * n} elements per rank")

if __name__ == "__main__":
    if len(sys.argv) < 5:
        print("Usage:")
        print("  python3 gen_data.py <rank_size> <m> <k> <n> [all_one]")
        print("Example:")
        print("  python3 gen_data.py 4 1024 1024 1024 all_one")
        print("\nNote: ReduceScatter output size = m*n/rank_size")
        print("       all_one mode: all inputs = 1, expected output = k * rank_size")
        sys.exit(1)

    rank_size = int(sys.argv[1])
    m = int(sys.argv[2])
    k = int(sys.argv[3])
    n = int(sys.argv[4])
    all_one = len(sys.argv) > 5 and sys.argv[5] == "all_one"
    
    gen_golden_data(rank_size, m, k, n, all_one)

**数据生成流程：**
1. 权重矩阵 B（所有 Rank 共享）：shape[K,N]，bfloat16，支持随机和全1模式
2. 输入矩阵 A（per rank）：shape[M,K]，不同种子生成不同数据
3. CPU Matmul 参考输出：`A.float() @ B.float()` → shape[M,N]
4. Golden ReduceScatter：对每个 Rank，取所有 Rank Matmul 输出的对应 chunk 求和

**全1测试模式**：输入全 1.0 时，Matmul 输出每元素 = K，ReduceScatter 后每元素 = K × rankSize。

### **11.2 精度验证**

In [ ]:
%%writefile Sources/01.03/mte_matmul_reduce_scatter/op_host/scripts/verify_result.py
import torch
import numpy as np
import sys
import os

def main():
    if len(sys.argv) != 5:
        print("Usage: python3 verify_result.py <rank_size> <m> <k> <n>")
        sys.exit(1)

    rank_size = int(sys.argv[1])
    m = int(sys.argv[2])
    k = int(sys.argv[3])
    n = int(sys.argv[4])

    rtol = 1e-2
    atol = 4e-2

    # ReduceScatter输出大小 = m*n / rank_size
    chunk_size = m * n // rank_size
    
    print(f"ReduceScatter Verification:")
    print(f"  rank_size: {rank_size}")
    print(f"  m*n per rank: {m * n}")
    print(f"  chunk_size (output per rank): {chunk_size}")
    print(f"  tolerance: rtol={rtol}, atol={atol}")

    all_match = True
    max_diff_all = 0.0
    total_passed = 0
    total_elements_all = 0

    for rank_id in range(rank_size):
        # 加载golden ReduceScatter输出
        golden_path = f"./output/cpu_reducescatter_rank{rank_id}.bin"
        if not os.path.exists(golden_path):
            print(f"ERROR: {golden_path} not found")
            sys.exit(1)

        golden_bytes = np.fromfile(golden_path, dtype=np.uint8)
        golden_tensor = torch.frombuffer(golden_bytes, dtype=torch.bfloat16)
        
        if len(golden_tensor) < chunk_size:
            golden_tensor = torch.nn.functional.pad(golden_tensor, (0, chunk_size - len(golden_tensor)))
        else:
            golden_tensor = golden_tensor[:chunk_size]
        
        golden_float = golden_tensor.float()

        # 加载NPU ReduceScatter输出
        npu_out_path = f"./output/npu_out_rank{rank_id}.bin"
        if not os.path.exists(npu_out_path):
            print(f"ERROR: {npu_out_path} not found")
            sys.exit(1)

        npu_bytes = np.fromfile(npu_out_path, dtype=np.uint8)
        npu_tensor = torch.frombuffer(npu_bytes, dtype=torch.bfloat16)

        if len(npu_tensor) < chunk_size:
            npu_tensor = torch.nn.functional.pad(npu_tensor, (0, chunk_size - len(npu_tensor)))
        else:
            npu_tensor = npu_tensor[:chunk_size]

        npu_float = npu_tensor.float()

        print(f"\n[Rank {rank_id}] Comparing NPU output with golden (size={chunk_size})")

        # 计算误差
        abs_diff = (npu_float - golden_float).abs()
        max_diff = abs_diff.max().item()
        mean_diff = abs_diff.mean().item()
        
        passed_mask = abs_diff <= (atol + rtol * golden_float.abs())
        passed_count = passed_mask.sum().item()
        failed_count = chunk_size - passed_count
        
        total_passed += passed_count
        total_elements_all += chunk_size
        
        pass_rate = passed_count / chunk_size * 100
        
        if failed_count > 0:
            all_match = False
            max_diff_all = max(max_diff_all, max_diff)
            
            failed_indices = torch.where(~passed_mask)[0].tolist()
            print(f"  pass_rate={pass_rate:.2f}% ({passed_count}/{chunk_size}), "
                  f"max_diff={max_diff:.6f}, mean_diff={mean_diff:.6f}, failed={failed_count}")
            print(f"    前10错误点:")
            for i, idx in enumerate(failed_indices[:10]):
                global_idx = rank_id * chunk_size + idx
                row = global_idx // n
                col = global_idx % n
                print(f"      [{i}] chunk_idx={idx} (global_idx={global_idx}, row={row}, col={col}), "
                      f"npu={npu_float[idx].item():.6f}, "
                      f"golden={golden_float[idx].item():.6f}, "
                      f"diff={abs_diff[idx].item():.6f}")
        else:
            print(f"  pass_rate=100.00% ({chunk_size}/{chunk_size}), max_diff={max_diff:.6f}, PASS")

    overall_pass_rate = total_passed / total_elements_all * 100
    print(f"\n[Overall] pass_rate={overall_pass_rate:.2f}% ({total_passed}/{total_elements_all})")

    if all_match:
        print(f"\n[PASS] All ranks ReduceScatter verified successfully")
    else:
        print(f"\n[FAIL] Verification failed, max_diff={max_diff_all:.6f}")
        sys.exit(1)

if __name__ == "__main__":
    main()

**精度标准：**
- `rtol = 1e-2`（相对误差 1%）
- `atol = 4e-2`（绝对误差 0.04）
- 逐元素判定：`|npu - golden| <= atol + rtol × |golden|`
- bfloat16 数据比对前转为 float32

### **11.3 编译运行命令**

In [ ]:
# 注意：需要在有 NPU 卡的环境中执行
# 步骤1: 编译
# cd Sources/01.03/mte_matmul_reduce_scatter && bash build.sh
# 步骤2: 运行测试
# cd Sources/01.03/mte_matmul_reduce_scatter/build && bash run_multi_data.sh

print("Build: cd Sources/01.03/mte_matmul_reduce_scatter && bash build.sh")
print("Run:   cd Sources/01.03/mte_matmul_reduce_scatter/build && bash run_multi_data.sh")

---
## **12. 开发流程总结**

回顾 MC2 融合算子开发的完整流程：

| 阶段 | 关键任务 | 输出物 |
|------|----------|--------|
| 1. 需求分析 | 确定融合算子组合、数据流、通信模式 | 计算流程图 |
| 2. Tiling设计 | 多核切分策略（Split-M）、UB Buffer规划 | Tiling结构体 |
| 3. Kernel实现 | Matmul计算块 + MTE通信模块 + 同步机制 | op_kernel/*.h/*.cpp |
| 4. Host实现 | 多设备管理、Tiling构造、Kernel Launch | op_host/*.cpp |
| 5. 测试验证 | Golden数据生成、多Case测试 | gen_data.py, verify_result.py |

**MC2 融合 vs 传统自定义算子：**

| 维度 | 传统自定义算子 | MC2 融合算子 |
|------|---------------|-------------|
| 调用方式 | aclnn 接口 | `<<<>>>` Kernel 直调 |
| 设备数量 | 单设备 | 多设备（需 HCCL） |
| 通信方式 | 无跨设备通信 | MTE Win Area / HCCL |
| 核类型 | 单类型 | AIC + AIV 混合 |

---
# **课后实践**

### **练习1：修改 Tiling 参数观察行为**
修改 `gMinSplitM` 的值（尝试 128、256），观察 splitMCnt 和 splitM 的计算结果变化。提示：关注 `CreateTilingDataAndContext` 中的校验和计算逻辑。

### **练习2：添加新的测试 Case**
在 `run_multi_data.sh` 的 `TEST_CASES` 数组中添加 `"custom_512:512:512:512"`，验证新尺寸下的正确性。

### **练习3：理解 Split-M 的设计意图**
思考：如果不做 Split-M（splitMCnt=1），会带来什么问题？从 Win Area 内存占用和同步粒度两个角度分析。

### **练习4：扩展思考**
如果要实现 AllGather+Matmul（PUT 模式）的 MC2 融合算子，与当前的 GET 模式在流程上有哪些不同？需要修改哪些模块？